<a href="https://colab.research.google.com/github/CrisEsp/Heur-stico/blob/secundario/Heuristico_13.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:

import random
import time
import threading
import sys
import threading
from queue import Queue

def redondear_diccionario(d, decimales=2):
    return {k: round(v, decimales) if isinstance(v, (int, float)) else v for k, v in d.items()}

# Variables globales
tipos_produccion_actual = {
    "MC1": None,
    "MC2": None,
    "MC3": None
}

# Variables globales para rastrear alimentaciones y temporizadores
alimentaciones_actuales = set()
alimentaciones_en_progreso = {}
#temporizadores_molinos = {}
lock = threading.Lock()
lock_print = threading.Lock()


def nivel_ineficiente_tolva(molino, material):
    if molino == "MC1":
        niveles = niveles_MC1
        tolvas = tolvas_MC1
        return 0.2*tolvas[material]['max_metros']  # 20% de la capacidad
    elif molino == "MC2":
        niveles = niveles_MC2
        tolvas = tolvas_MC2
        return 0.2*tolvas[material]['max_metros']  # 20% de la capacidad
    # else:  # MC3
    #     niveles = niveles_MC3
    #     tolvas = tolvas_MC3
    #     return 0.5*tolvas[material]['max_porcentaje'] # 50% de la capacidad
    elif molino == "MC3":
        if material == "Clinker_Silo_Blanco":
            niveles = niveles_MC3
            tolvas = tolvas_MC3
            return 0.5*tolvas[material]['max_metros']  # 50% de la capacidad
        else:
            niveles = niveles_MC3
            tolvas = tolvas_MC3
            return 0.5*tolvas[material]['max_porcentaje'] # 50% de la capacidad

# Cola global para materiales pendientes de alimentación
cola_alimentacion = Queue()

# Definición de capacidades de tolvas y niveles iniciales
tolvas_MC1 = {
    "Clinker": {"capacidad": 500, "max_metros": 14},
    "Puzolana_Humeda": {"capacidad": 300, "max_metros": 12},
    "Yeso": {"capacidad": 300, "max_metros": 10}
}
tolvas_MC2 = {
    "Clinker": {"capacidad": 300, "max_metros": 9},
    "Puzolana_Humeda": {"capacidad": 500, "max_metros": 15, "tolva": "426HO04"},
    "Puzolana_Seca": {"capacidad": 100, "max_metros": 12, "tolva": "426HO02"},
    "Yeso": {"capacidad": 120, "max_metros": 9}
}
tolvas_MC3 = {
    "Clinker": {"capacidad": 60, "max_porcentaje": 100},
    "Clinker_Silo_Blanco": {"capacidad": 500, "max_metros": 10.5},   #AQUI ES EN MTROS
    "Puzolana_Seca": {"capacidad": 35, "max_porcentaje": 100},
    "Yeso": {"capacidad": 30, "max_porcentaje": 100}
}

# Definir los límites basados en los valores máximos de cada tolva
limites_tolvas = {
    "MC1": {
        "Clinker": {"unidad": "metros", "min": 0, "max": tolvas_MC1["Clinker"]["max_metros"]},
        "Puzolana_Humeda": {"unidad": "metros", "min": 0, "max": tolvas_MC1["Puzolana_Humeda"]["max_metros"]},
        "Yeso": {"unidad": "metros", "min": 0, "max": tolvas_MC1["Yeso"]["max_metros"]}
    },
    "MC2": {
        "Clinker": {"unidad": "metros", "min": 0, "max": tolvas_MC2["Clinker"]["max_metros"]},
        "Puzolana_Humeda": {"unidad": "metros", "min": 0, "max": tolvas_MC2["Puzolana_Humeda"]["max_metros"]},
        "Puzolana_Seca": {"unidad": "metros", "min": 0, "max": tolvas_MC2["Puzolana_Seca"]["max_metros"]},
        "Yeso": {"unidad": "metros", "min": 0, "max": tolvas_MC2["Yeso"]["max_metros"]}
    },
    "MC3": {
        "Clinker": {"unidad": "%", "min": 0, "max": tolvas_MC3["Clinker"]["max_porcentaje"]},
        "Clinker_Silo_Blanco": {"unidad": "metros", "min": 0, "max": tolvas_MC3["Clinker_Silo_Blanco"]["max_metros"]},
        "Puzolana_Seca": {"unidad": "%", "min": 0, "max": tolvas_MC3["Puzolana_Seca"]["max_porcentaje"]},
        "Yeso": {"unidad": "%", "min": 0, "max": tolvas_MC3["Yeso"]["max_porcentaje"]}
    }
}

# Niveles ingresados por el usuario en metros para MC1 y MC2 ;para MC3 se mide en % excepto Clinker Silo blanco que se mide en metros
# niveles_MC1 = {"Clinker": 4.5, "Puzolana_Humeda": 7.4, "Yeso":1.8}
# niveles_MC2 = {"Clinker": 5.2, "Puzolana_Humeda": 11, "Puzolana_Seca": 7.4, "Yeso": 6.5}
# niveles_MC3 = {"Clinker": 78, "Clinker_Silo_Blanco": 5.5, "Puzolana_Seca":64, "Yeso": 47}

# Niveles ingresados por el usuario en metros para MC1 y MC2 ;para MC3 se mide en % excepto Clinker Silo blanco que se mide en metros
niveles_MC1 = {"Clinker": 0, "Puzolana_Humeda": 0, "Yeso":9}
niveles_MC2 = {"Clinker": 5, "Puzolana_Humeda": 11, "Puzolana_Seca": 7, "Yeso": 6}
niveles_MC3 = {"Clinker":78, "Clinker_Silo_Blanco": 11, "Puzolana_Seca":64, "Yeso":0 }


def actualizar_niveles_y_calcular_toneladas(niveles_MC1, niveles_MC2, niveles_MC3, tolvas_MC1, tolvas_MC2, tolvas_MC3):
    print("--- Actualización de niveles y cálculo de toneladas ---")

    # Calcular toneladas para cada molino
    toneladas_MC1 = metros_a_toneladas(niveles_MC1, tolvas_MC1)
    toneladas_MC2 = metros_a_toneladas(niveles_MC2, tolvas_MC2)
    toneladas_MC3 = metros_a_toneladas(niveles_MC3, tolvas_MC3)

    # Imprimir los resultados
    print("Niveles actuales en metros")
    print(f"Niveles MC1 = {niveles_MC1}")
    print(f"Niveles MC2 = {niveles_MC2}")
    print(f"Niveles MC3 = {niveles_MC3}")
    print("")

    print("Niveles actuales en toneladas")
    print(f"Niveles MC1 toneladas = {toneladas_MC1}")
    print(f"Niveles MC2 toneladas = {toneladas_MC2}")
    print(f"Niveles MC3 toneladas = {toneladas_MC3}")
    print("")

def solicitar_niveles_iniciales():
    """
    Solicitar al operador que ingrese los niveles iniciales de los materiales
    para los molinos MC1, MC2 y MC3.
    """
    print("Ingrese los niveles iniciales de los materiales para cada molino:")

    # Solicitar niveles para MC1
    print("\nMolino MC1:")
    niveles_MC1["Clinker"] = float(input("Nivel de Clinker (metros): "))
    niveles_MC1["Puzolana_Humeda"] = float(input("Nivel de Puzolana (metros): "))
    niveles_MC1["Yeso"] = float(input("Nivel de Yeso (metros): "))

    # Solicitar niveles para MC2
    print("\nMolino MC2:")
    niveles_MC2["Clinker"] = float(input("Nivel de Clinker (metros): "))
    niveles_MC2["Puzolana_Humeda"] = float(input("Nivel de Puzolana Húmeda (metros): "))
    niveles_MC2["Puzolana_Seca"] = float(input("Nivel de Puzolana Seca (metros): "))
    niveles_MC2["Yeso"] = float(input("Nivel de Yeso (metros): "))

    # Solicitar niveles para MC3
    print("\nMolino MC3:")
    niveles_MC3["Clinker"] = float(input("Nivel de Clinker (%): "))
    niveles_MC3["Clinker_Silo_Blanco"] = float(input("Nivel de Clinker Silo Blanco (metros): "))
    niveles_MC3["Puzolana_Seca"] = float(input("Nivel de Puzolana (%): "))
    niveles_MC3["Yeso"] = float(input("Nivel de Yeso (%): "))



def ingresar_niveles(molino):
    """Solicita al operador que ingrese los niveles actuales para cada material del molino, con la opción de mantener el valor actual."""
    niveles = {}
    tolvas = limites_tolvas[molino]
    for material, propiedades in tolvas.items():
        unidad = propiedades["unidad"]
        min_nivel = propiedades["min"]
        max_nivel = propiedades["max"]

        # Obtener el nivel actual desde el diccionario de niveles para el molino
        if molino == "MC1":
            nivel_actual = niveles_MC1.get(material, 0)
        elif molino == "MC2":
            nivel_actual = niveles_MC2.get(material, 0)
        elif molino == "MC3":
            nivel_actual = niveles_MC3.get(material, 0)

        # Mostrar el nivel actual y permitir que el operador lo mantenga si presiona "Enter"
        while True:
            try:
                # Formatear la entrada con el cálculo ya realizada
                entrada = input(f"Ingrese el nivel actual de {material} en {molino} ({min_nivel} hasta {max_nivel} {unidad}) (actual: {nivel_actual} {unidad}) o presione Enter para mantener el valor actual: ")
                if entrada == "":  # Si se presiona "Enter", mantener el nivel actual
                    niveles[material] = nivel_actual
                    break
                else:
                    nivel = float(entrada)  # Convertir el valor ingresado a float
                    # Validar que el nivel esté dentro de los límites
                    if nivel < min_nivel or nivel > max_nivel:
                        print(f"Error: El nivel de {material} debe estar entre {min_nivel} y {max_nivel} {unidad}.")
                    else:
                        niveles[material] = nivel
                        break
            except ValueError:
                print("Error: Por favor, ingrese un valor numérico válido.")

    return niveles


def metros_a_toneladas(niveles, tolvas):
    toneladas = {}

    for material, metros in niveles.items():
        # Verificar que el material existe en el diccionario de tolvas
        if material in tolvas:
            tolva_material = tolvas[material]

            # Verificar si se trabaja con metros o porcentaje
            if "max_metros" in tolva_material:
                capacidad = tolva_material["capacidad"]
                max_metros = tolva_material["max_metros"]
                # Calcular toneladas basadas en metros
                toneladas[material] = round((metros * capacidad) / max_metros, 2)
            elif "max_porcentaje" in tolva_material:
                capacidad = tolva_material["capacidad"]
                # Calcular toneladas basadas en porcentaje
                toneladas[material] = round((metros * capacidad) / 100, 2)
            else:
                # Si no hay información suficiente para calcular
                print(f"Advertencia: No se puede calcular para {material}, faltan datos en la tolva.")
        else:
            # Material no está en el diccionario de tolvas
            print(f"Advertencia: {material} no se encuentra en el diccionario de tolvas.")

    return toneladas

# Convertimos los niveles de metros a toneladas
niveles_MC1_toneladas = metros_a_toneladas(niveles_MC1, tolvas_MC1)
niveles_MC2_toneladas = metros_a_toneladas(niveles_MC2, tolvas_MC2)
niveles_MC3_toneladas = metros_a_toneladas(niveles_MC3, tolvas_MC3)



# Diccionario con el estado de los molinos
estado_molinos = {
    "MC1": True,  # True = en marcha, False = detenido
    "MC2": True,
    "MC3": True
}

def verificar_estado_molino(molino):
    """
    Función que verifica si un molino está en marcha o detenido.
    Devuelve True si el molino está en marcha, False si está detenido.
    """
    if molino in estado_molinos:
        return estado_molinos[molino]
    else:
        print(f"Error: Estado desconocido para el molino {molino}")
        return False


def obtener_tipo_produccion_actual(molino):
    """    Retorna el tipo de producción actual para un molino específico basado en lo que el usuario haya ingresado.
    """
    return tipos_produccion_actual.get(molino, None)  # Retorna el tipo de producción si está presente, o None si no se ha ingresado

def calcular_consumo(molino, material):
    tipo_produccion = obtener_tipo_produccion_actual(molino)
    necesidades = calcular_necesidades(molino, tipo_produccion)
    return necesidades.get(material, 0)


def calcular_necesidades(molino, tipo_produccion):        # Funcion actualizada : 08/10/24
    # Definiciones de producción por tipo
    produccion_MC1 = {
        "P30": (0.30, 0.015, 75),
        "P40": (0.40, 0.015, 65)
    }
    produccion_MC2 = {
        "P10": (0.10, 0.03, 65),
        "P16": (0.16, 0.025, 80),
        "P20": (0.12, 0.025, 90),
        "P30": (0.30, 0.02, 110)
    }
    produccion_MC3 = {
        "P30": (0.30, 0.025, 36)
    }

    # Verificar si el molino está encendido
    if not verificar_estado_molino(molino):
        print(f"El molino {molino} está detenido. No se calcularán las necesidades.")
        return {}  # Retornar un diccionario vacío si el molino está apagado

    # Calcular necesidades si el molino está encendido
    if molino == "MC1" and tipo_produccion in produccion_MC1:
        puzolana, yeso, produccion = produccion_MC1[tipo_produccion]
        clinker = 1 - puzolana - yeso
        return {
            "Clinker": clinker * produccion,
            "Puzolana_Humeda": puzolana * produccion,
            "Yeso": yeso * produccion
        }
    elif molino == "MC2" and tipo_produccion in produccion_MC2:
        puzolana, yeso, produccion = produccion_MC2[tipo_produccion]
        clinker = 1 - puzolana - yeso
        return {
            "Clinker": clinker * produccion,
            "Puzolana_Humeda": puzolana * produccion,
            "Puzolana_Seca": puzolana * produccion,
            "Yeso": yeso * produccion
        }
    elif molino == "MC3" and tipo_produccion in produccion_MC3:
        puzolana, yeso, produccion = produccion_MC3[tipo_produccion]
        clinker = 1 - puzolana - yeso
        return {
            "Clinker": clinker * produccion,
            "Puzolana_Seca": puzolana * produccion,
            "Clinker_Silo_Blanco": clinker * produccion,
            "Yeso": yeso * produccion
        }

    print(f"Advertencia: No se encontró configuración para {molino} con producto {tipo_produccion}")
    print("")
    return {}  # Retornar un diccionario vacío si no hay coincidencias

# Función de cálculo de tiempo de vaciado
def calcular_tiempo_vaciado(molino, material):
    # Verificamos si el molino está en marcha
    if not verificar_estado_molino(molino):
        print(f"El molino {molino} está detenido. No se realizarán cálculos para este molino.")
        return None

    # Continuamos con los cálculos solo para los molinos en marcha
    if molino == "MC1":
        niveles = niveles_MC1
        tolvas = tolvas_MC1
    elif molino == "MC2":
        niveles = niveles_MC2
        tolvas = tolvas_MC2
    elif molino == "MC3":
        niveles = niveles_MC3
        tolvas = tolvas_MC3
    else:
        print(f"Molino {molino} no reconocido.")
        return None

    # Verificar si el material existe en las tolvas
    if material not in tolvas:
        print(f"Advertencia: El material '{material}' no se encuentra configurado en las tolvas de {molino}.")
        return None

    capacidad = tolvas[material]['capacidad']
    nivel_actual = niveles.get(material, 0)
    nivel_ineficiente = nivel_ineficiente_tolva(molino, material)

    # Calcular las toneladas reales basadas en el tipo de medición
    if 'max_metros' in tolvas[material]:
        max_nivel = tolvas[material]['max_metros']
        toneladas_reales = ((nivel_actual - nivel_ineficiente) * capacidad) / max_nivel
    else:  # Para MC3 que usa porcentajes
        toneladas_reales = ((nivel_actual - nivel_ineficiente) * capacidad) / 100

    # Calcular el consumo
    consumo = calcular_consumo(molino, material)

    # Verificar si el consumo es válido
    if consumo is None or consumo <= 0:
        #print(f"Advertencia: No se encontró configuración válida para el consumo de {material} en {molino} o el consumo es 0")
        return float('inf')

    #print(f"Consumo de {material} en {molino}: {consumo:.2f}t/h ")
    return toneladas_reales / consumo


# ***** RESTRICCIONES PARA MC1 ******************
def nivel_silo_blanco():
     return ((niveles_MC3["Clinker_Silo_Blanco"] - tolvas_MC3["Clinker_Silo_Blanco"]["max_metros"]*0.2)* tolvas_MC3["Clinker_Silo_Blanco"]["capacidad"]) / tolvas_MC3["Clinker_Silo_Blanco"]["max_metros"]
     print("nivel_silo_blanco",nivel_silo_blanco())

def nivel_426HO04():
    return ((niveles_MC2["Puzolana_Humeda"] - tolvas_MC2["Puzolana_Humeda"]["max_metros"]*0.2)*tolvas_MC2["Puzolana_Humeda"]["capacidad"]) / tolvas_MC2["Puzolana_Humeda"]["max_metros"]

def nivel_puzolana_humeda():
    return ((niveles_MC2["Puzolana_Humeda"] - tolvas_MC2["Puzolana_Humeda"]["max_metros"]*0.2)*tolvas_MC2["Puzolana_Humeda"]["capacidad"]) / tolvas_MC2["Puzolana_Humeda"]["max_metros"]

def se_alimenta_puzolana_a_L1():
    return ("MC1", "Puzolana_Humeda") in alimentaciones_actuales

def se_alimenta_yeso_a_L1():
    return ("MC1", "Yeso") in alimentaciones_actuales

def se_desea_producir_P10():
    return obtener_tipo_produccion_actual("MC2") == "P10"


def se_alimenta_yeso_a_L1_por_L1():
    return ("MC1", "Yeso") in alimentaciones_actuales and alimentaciones_en_progreso.get("MC1", set()).intersection({"Yeso"})

def se_alimenta_clinker_a_Silo_Blanco_L3():
    return ("MC3", "Clinker_Silo_Blanco") in alimentaciones_actuales

def se_puede_alimentar_L1_por_L2():
    return not any(alim[0] == "MC2" for alim in alimentaciones_actuales)

def se_requiere_alimentar_L2(niveles_MC2):
    # Definir materiales críticos que se requieren para alimentar L2
    materiales_criticos = ['clinker', 'yeso', 'puzolana']

    # Revisar si algún material crítico tiene un nivel bajo
    if any(niveles_MC2[material] < 10 for material in materiales_criticos if material in niveles_MC2):
        print("Se requiere alimentar L2 debido a niveles bajos de materiales.")
        return True

    # Comprobar si se está produciendo un producto que requiere alimentación en L2
    if se_desea_producir_P10():  # Asumiendo que esta función ya está definida
        print("Se requiere alimentar L2 porque se desea producir P10.")
        return True

    # Si ninguna de las condiciones anteriores se cumple
    print("No se requiere alimentar L2 en este momento.")
    return False

def se_alimenta_yeso_a_L2_por_L2():
    return ("MC2", "Yeso") in alimentaciones_actuales

def se_alimenta_yeso_a_L3_por_L2():
    return ("MC3", "Yeso") in alimentaciones_actuales and ("MC2", "Yeso") in alimentaciones_actuales

def se_alimenta_yeso_a_L1_por_L2():
    return ("MC1", "Yeso") in alimentaciones_actuales and ("MC2", "Yeso") in alimentaciones_actuales

def se_alimenta_puzolana_a_L1_por_L2():
    return ("MC1", "Puzolana_Humeda") in alimentaciones_actuales and ("MC2", "Puzolana_Humeda") in alimentaciones_actuales

def se_alimenta_PH_a_426HO04_por_L2():
    return ("MC2", "Puzolana_Humeda") in alimentaciones_actuales

def se_alimenta_yeso_a_L3():
    return ("MC3", "Yeso") in alimentaciones_actuales

def requiere_alimentacion_puzolana_humeda(molino):
    if molino == "MC1":
        return niveles_MC1["Puzolana_Humeda"] < tolvas_MC1["Puzolana_Humeda"]["max_metros"] * 0.8
    elif molino == "MC2":
        return niveles_MC2["Puzolana_Humeda"] < tolvas_MC2["Puzolana_Humeda"]["max_metros"] * 0.8
    return False

def requiere_alimentacion_puzolana_seca(molino):
    if molino == "MC2":
        return niveles_MC2["Puzolana_Seca"] < tolvas_MC2["Puzolana_Seca"]["max_metros"] * 0.8
    elif molino == "MC3":
        return niveles_MC3["Puzolana_Seca"] < tolvas_MC3["Puzolana"]["max_porcentaje"] * 0.5
    return False

def ajustar_velocidad_transportador(min_vel, max_vel):
    velocidad = random.uniform(min_vel, max_vel)
    print(f"Ajustando velocidad del transportador a {velocidad:.2f} RPM")
    return velocidad

def se_puede_alimentar_por_L1():
    return not any(alim[0] == "MC1" for alim in alimentaciones_actuales)

def puede_alimentar_yeso_a_L1():
    nivel_actual = niveles_MC1["Yeso"]
    max_nivel = min(tolvas_MC1["Yeso"]["max_metros"], 2)  # Limitar a 2m para yeso en MC1
    return nivel_actual < max_nivel

def se_alimenta_puzolana():
    return any([
        #("MC1", "Puzolana_Humeda") in alimentaciones_actuales,
        ("MC2", "Puzolana_Humeda") in alimentaciones_actuales,
        ("MC2", "Puzolana_Seca") in alimentaciones_actuales,
        ("MC3", "Puzolana_Seca") in alimentaciones_actuales
    ])


def seleccionar_ruta_alimentacion(molino, material,material_critico):
    if material == "Clinker" or material=="Clinker_Silo_Blanco":
        return seleccionar_ruta_clinker(molino,material)
    elif material in ["Puzolana_Humeda", "Puzolana_Seca"]:
        return seleccionar_ruta_puzolana(molino, material,material_critico,material)
    elif material == "Yeso":
        return seleccionar_ruta_yeso(molino,material_critico)
    else:
        return "Material no reconocido"

def seleccionar_ruta_clinker(molino,material):
    print(f"\n--- Seleccionando ruta de Clinker para {molino} ---")

    # Verificar si el molino está en marcha, pero las rutas de alimentación se mantienen disponibles
    if not verificar_estado_molino(molino):
        print(f"El molino {molino} está detenido, pero las rutas de alimentación están disponibles.")
        return f"El molino {molino} está detenido"

    # Verificar el estado de las tolvas antes de seleccionar la ruta
    estado_tolvas = revisar_tolvas_y_evitar_alimentacion(molino)

    # Si la tolva de Puzolana está llena, no se selecciona una ruta de alimentación
    if estado_tolvas[molino].get("Clinker", False):
        print(f"Tolva de Clinker en {molino} está llena. No se seleccionará ruta de alimentación.")
        return f"No se puede alimentar más Clinker en {molino}, la tolva está llena --"

    # Calcular tiempos de vaciado solo para molinos que están en marcha
    tiempo_vaciado_L1 = calcular_tiempo_vaciado("MC1", "Clinker") if verificar_estado_molino("MC1") else None
    tiempo_vaciado_L2 = calcular_tiempo_vaciado("MC2", "Clinker") if verificar_estado_molino("MC2") else None
    tiempo_vaciado_L3 = calcular_tiempo_vaciado("MC3", "Clinker") if verificar_estado_molino("MC3") else None
    tiempo_vaciado_SB_L3 = calcular_tiempo_vaciado("MC3", "Clinker_Silo_Blanco") if verificar_estado_molino("MC3") else None

    # Verifica si los tiempos son None y ajusta el mensaje
    tiempo_vaciado_L1_str = f"{tiempo_vaciado_L1:.2f}h" if tiempo_vaciado_L1 is not None else "Molino apagado"
    tiempo_vaciado_L2_str = f"{tiempo_vaciado_L2:.2f}h" if tiempo_vaciado_L2 is not None else "Molino apagado"
    tiempo_vaciado_L3_str = f"{tiempo_vaciado_L3:.2f}h" if tiempo_vaciado_L3 is not None else "Molino apagado"
    tiempo_vaciado_SB_L3_str = f"{tiempo_vaciado_SB_L3:.2f}h" if tiempo_vaciado_SB_L3 is not None else "Molino apagado"

    # tiempo_vaciado_yeso = calcular_tiempo_vaciado("MC1", "Yeso") if verificar_estado_molino("MC1") else None
    # tiempo_vaciado_puzolana = calcular_tiempo_vaciado("MC1", "Puzolana_Humeda") if verificar_estado_molino("MC1") else None

    # # Verificar si los tiempos son None y ajusta el mensaje
    # tiempo_vaciado_yeso_str = f"{tiempo_vaciado_yeso:.2f}h" if tiempo_vaciado_yeso is not None else "Molino apagado"
    # tiempo_vaciado_puzolana_str = f"{tiempo_vaciado_puzolana:.2f}h" if tiempo_vaciado_puzolana is not None else "Molino apagado"

    # # Manejo de tiempos de vaciado negativos
    # if tiempo_vaciado_yeso is not None and tiempo_vaciado_yeso < 0:
    #     print("Advertencia: Tiempo de vaciado de Yeso es negativo, posiblemente el silo está vacío")
    # if tiempo_vaciado_puzolana is not None and tiempo_vaciado_puzolana < 0:
    #     print("Advertencia: Tiempo de vaciado de Puzolana_Humeda es negativo, posiblemente el silo está vacío")


    # Imprime los tiempos de vaciado
    print(f"Tiempos de vaciado: L1={tiempo_vaciado_L1_str}, L2={tiempo_vaciado_L2_str}, L3={tiempo_vaciado_L3_str}")

    nivel_actual_silo_blanco = niveles_MC3["Clinker_Silo_Blanco"]


    # Manejo de tiempos de vaciado negativos (verificación en las variables numéricas, no en las cadenas)
    if tiempo_vaciado_L1 is not None and tiempo_vaciado_L1 < 0:
        print("Advertencia: Tiempo de vaciado L1 es negativo, posiblemente el silo está vacío")
    if tiempo_vaciado_L2 is not None and tiempo_vaciado_L2 < 0:
        print("Advertencia: Tiempo de vaciado L2 es negativo, posiblemente el silo está vacío")
    if tiempo_vaciado_L3 is not None and tiempo_vaciado_L3 < 0:
        print("Advertencia: Tiempo de vaciado L3 es negativo, posiblemente el silo está vacío")

    # Verificar si al menos uno de los molinos tiene un tiempo de vaciado válido
    if tiempo_vaciado_L1 is not None or tiempo_vaciado_L2 is not None or tiempo_vaciado_L3 is not None:
        # Procesar los molinos que están en marcha
        if tiempo_vaciado_L1 is not None:
          if molino == "MC1":
              # # Selección del material más crítico para MC1
              # tiempos_materiales = {
              #     "Yeso": tiempo_vaciado_yeso,
              #     "Puzolana_Humeda": tiempo_vaciado_puzolana
              # }
              # material_critico = min(tiempos_materiales, key=lambda m: tiempos_materiales[m] if tiempos_materiales[m] is not None else float('inf'))

              if tiempo_vaciado_L1 is None or tiempo_vaciado_L1 < 0:
                  print("Forzar alimentación a MC1 desde pretrit o recargar Clinker")
                  return "Hacia MC1 desde pretrit"

              else:
                  # Verificar que no haya conflicto con la alimentación de Yeso o Puzolana en MC1
                  # if "Yeso" in alimentaciones_en_progreso.get("MC1", []) or "Puzolana_Humeda" in alimentaciones_en_progreso.get("MC1", []):
                  #     print("No se puede alimentar Clinker en MC1 mientras se alimenta Yeso o Puzolana.")
                  #     return "No se puede alimentar Clinker en este momento"
                  if tiempo_vaciado_L1 > max(tiempo_vaciado_L2 or -1, tiempo_vaciado_L3 or -1):
                      print("MC1 tiene el mayor tiempo de vaciado, alimentar desde pretrit")
                      return "Hacia MC1 desde pretrit"
                  else:
                      if nivel_actual_silo_blanco > 3:
                          print("Alimentar Clinker a MC3 desde silo blanco")
                          return "Hacia MC3 desde Silo Blanco"

        # if tiempo_vaciado_L2 is not None:
        #   # Si el molino objetivo es MC2
        #   #elif molino == "MC2":
        #       if tiempo_vaciado_L2 is None or tiempo_vaciado_L2 < 0:
        #           print("Forzar alimentación a MC2 desde pretrit o recargar Clinker")
        #           return "Forzar alimentación a MC2 desde pretrit"
        #       else:
        #           if tiempo_vaciado_L2 > max(tiempo_vaciado_L1 or -1, tiempo_vaciado_L3 or -1):
        #               print("MC2 tiene el mayor tiempo de vaciado que MC3, alimentar desde pretrit")
        #               #return "Hacia MC2 desde pretrit"
        #               if nivel_actual_silo_blanco > 3:
        #                   return "Alimentar Clinker a MC3 desde Silo Blanco"
        #               else:
        #                   return "Alimentar Clinker a Silo Blanco"
        #           else:
        #               print("MC3 tiene el mayor tiempo de vaciado que MC2, alimentar desde pretrit")
        #               return "Hacia MC2 desde pretrit"

        if tiempo_vaciado_L2 is not None:
          if molino== "MC2":
                if tiempo_vaciado_L2 is not None or tiempo_vaciado_L2 < 0:
                    print("Alimentación a MC2 desde pretrit")
                    return "Alimentar Clinker a MC2 desde pretrit"
                else:
                    # Asegurarse de que MC3 esté encendido antes de sugerir alimentación a MC3
                    if tiempo_vaciado_L2 > max(tiempo_vaciado_L1 or -1, tiempo_vaciado_L3 or -1):
                        print("MC2 tiene el mayor tiempo de vaciado que MC3, alimentar desde pretrit")
                        # Verificar que MC3 esté encendido antes de sugerir ruta hacia MC3
                        if nivel_actual_silo_blanco > 3 and verificar_estado_molino("MC3"):
                            return "Alimentar Clinker a MC3 desde Silo Blanco"
                        else:
                            if verificar_estado_molino("MC3") == True:
                              return "Alimentar Clinker a Silo Blanco"
                    else:
                        print("MC3 tiene el mayor tiempo de vaciado que MC2, alimentar desde pretrit")
                        return "Hacia MC2 desde pretrit"

        if tiempo_vaciado_L3 is not None or tiempo_vaciado_SB_L3 is not None:
          # Si el molino objetivo es MC3
           if molino == "MC3":
            if material =="Clinker":
               print("Buscando ruta para Clinker en MC3..")
               #if tiempo_vaciado_L3 > max(tiempo_vaciado_L1 or -1, tiempo_vaciado_L2 or -1):
               if tiempo_vaciado_L2 > tiempo_vaciado_L3:
                  if nivel_actual_silo_blanco > 3:
                      print("MC3 tiene el mayor tiempo de vaciado, alimentar desde silo blanco")
                      return "Hacia MC3 desde Silo blanco"
                  else:
                          print("MC3 tiene el mayor tiempo de vaciado, pero el silo blanco está bajo, alimentar hacia el silo blanco")
                          return "Alimentar Clinker a Silo Blanco, Nivel menor a 3 metros"
               elif molino=="MC2":
                    return "Alimentar Clinker a MC2 desde pretrit"


            if material == "Clinker_Silo_Blanco":
              if tiempo_vaciado_SB_L3 is None or tiempo_vaciado_SB_L3 < 0:
                  print("Forzar alimentación a MC3 desde pretrit o recargar Clinker")
                  return "Alimentación a Clinker en Silo Blanco"
              else:
                      if nivel_actual_silo_blanco > 3:
                          print("MC3 tiene el mayor tiempo de vaciado, alimentar desde silo blanco")
                          return "No se requiere alimentar a Silo Blanco"
                      else:
                          if material == "Clinker_Silo_Blanco":

                             print("MC3 tiene el mayor tiempo de vaciado, pero el silo blanco está bajo, alimentar hacia el silo blanco")
                             return "Alimentar Clinker a Silo Blanco"

              # if tiempo_vaciado_SB_L3 is None or tiempo_vaciado_SB_L3 < 0:
              #     print("Se nesita alimentar Clinker a Silo Blanco")
              #     return "Alimentar clinker a Silo Blanco"
              # else:
              #         if nivel_actual_silo_blanco > 3:
              #             print("MC3 tiene el mayor tiempo de vaciado, alimentar desde silo blanco")
              #             return "Hacia MC3 desde Silo blanco"
              #         else:
              #             print("MC3 tiene el mayor tiempo de vaciado, pero el silo blanco está bajo, alimentar hacia el silo blanco")
              #             return "Alimentar Clinker a Silo Blanco"

                  # else:
                  #     if nivel_actual_silo_blanco > 3:
                  #         print("Alimentar Clinker a MC3 desde silo blanco")
                  #         return "Hacia MC3 desde Silo Blanco"
                  #     else:
                  #         print("Silo blanco bajo, alimentar Clinker hacia el silo blanco")
                  #         return "Alimentar Clinker a Silo Blanco"

        # print("No se encontró una ruta válida para alimentar Clinker")
        # return "No se puede alimentar Clinker en este momento"
    else:
        print("No es posible comparar tiempos de vaciado debido a un molino apagado.")
    return "No se puede alimentar Clinker en este momento ---*"

# def seleccionar_ruta_puzolana(molino, tipo_puzolana,material_critico):
#     print(f"\n--- Seleccionando ruta de Puzolana para {molino} ---")


#     # Verificar el estado de las tolvas antes de seleccionar la ruta
#     estado_tolvas = revisar_tolvas_y_evitar_alimentacion()

#     # Si la tolva de Puzolana está llena, no se selecciona una ruta de alimentación
#     if estado_tolvas[molino].get(tipo_puzolana, False):
#         print(f"Tolva de {tipo_puzolana} en {molino} está llena. No se seleccionará ruta de alimentación.")
#         return f"No se puede alimentar más {tipo_puzolana} en {molino}, la tolva está llena."



#     # Calcular tiempos de vaciado solo si los molinos están en marcha
#     tiempo_vaciado_PH_L1 = calcular_tiempo_vaciado("MC1", "Puzolana_Humeda") if verificar_estado_molino("MC1") else None
#     tiempo_vaciado_PH_L2 = calcular_tiempo_vaciado("MC2", "Puzolana_Humeda") if verificar_estado_molino("MC2") else None
#     # Calcular tiempos de vaciado solo si los molinos están en marcha
#     tiempo_vaciado_PS_L2 = calcular_tiempo_vaciado("MC2", "Puzolana_Seca") if verificar_estado_molino("MC2") else None
#     tiempo_vaciado_PS_L3 = calcular_tiempo_vaciado("MC3", "Puzolana_Seca") if verificar_estado_molino("MC3") else None

#     # Verifica si los tiempos son None y ajusta el mensaje
#     tiempo_vaciado_PH_L1_str = f"{tiempo_vaciado_PH_L1:.2f}h" if tiempo_vaciado_PH_L1 is not None else "Molino apagado"
#     tiempo_vaciado_PH_L2_str = f"{tiempo_vaciado_PH_L2:.2f}h" if tiempo_vaciado_PH_L2 is not None else "Molino apagado"
#     tiempo_vaciado_PS_L2_str = f"{tiempo_vaciado_PS_L2:.2f}h" if tiempo_vaciado_PS_L2 is not None else "Molino apagado"
#     tiempo_vaciado_PS_L3_str = f"{tiempo_vaciado_PS_L3:.2f}h" if tiempo_vaciado_PS_L3 is not None else "Molino apagado"

#     # Manejo de tiempos de vaciado negativos (verificación en las variables numéricas)
#     if tiempo_vaciado_PH_L1 is not None and tiempo_vaciado_PH_L1 < 0:
#         print("Advertencia: Tiempo de vaciado L1 es negativo, posiblemente el silo está vacío")
#     if tiempo_vaciado_PH_L2 is not None and tiempo_vaciado_PH_L2 < 0:
#         print("Advertencia: Tiempo de vaciado L2 es negativo, posiblemente el silo está vacío")
#     if tiempo_vaciado_PS_L2 is not None and tiempo_vaciado_PS_L2 < 0:
#         print("Advertencia: Tiempo de vaciado L2 es negativo, posiblemente el silo está vacío")
#     if tiempo_vaciado_PS_L3 is not None and tiempo_vaciado_PS_L3 < 0:
#         print("Advertencia: Tiempo de vaciado L3 es negativo, posiblemente el silo está vacío")


#     # # Priorizar Yeso en MC1 si el material crítico no es Puzolana Húmeda
#     # if molino == "MC1" and material_critico != "Puzolana_Humeda":
#     #     print(f"Material crítico en {molino} no es Puzolana Húmeda, priorizando alimentación de Yeso.")
#     #     return "No se puede alimentar puzolana en este momento ,se está alimentando Yeso en MC1"

#     # Ruta para Puzolana Húmeda
#     if tipo_puzolana == "Puzolana_Humeda":
#         # Imprime los tiempos de vaciado
#         print(f"Tiempos de vaciado: PH L1={tiempo_vaciado_PH_L1_str}, PH L2={tiempo_vaciado_PH_L2_str}")
#         if tiempo_vaciado_PH_L1 is not None or tiempo_vaciado_PH_L2 is None:
#             print("Alimentación a MC1")
#             return "P.H a MC1 por MC1"
#         # Verificar cuál molino está en marcha para seleccionar la mejor ruta
#         if tiempo_vaciado_PH_L1 is not None or tiempo_vaciado_PH_L2 is not None:
#             if tiempo_vaciado_PH_L1 is not None and (tiempo_vaciado_PH_L2 is None or tiempo_vaciado_PH_L1 > tiempo_vaciado_PH_L2):
#                 if niveles_MC2["Puzolana_Humeda"] <= 9:
#                     if se_requiere_alimentar_L2(niveles_MC2):
#                         if not se_alimenta_yeso_a_L2_por_L2() and not se_alimenta_yeso_a_L3_por_L2() and not se_alimenta_puzolana_a_L1_por_L2():
#                             return "P.H. a 426HO04 por MC2"
#                     else:
#                         if se_puede_alimentar_L1_por_L2():
#                             return "Húmeda a MC1 por MC2"
#             elif tiempo_vaciado_PH_L2 is not None and (tiempo_vaciado_PH_L1 is None or tiempo_vaciado_PH_L2 >= tiempo_vaciado_PH_L1):
#                 if not se_alimenta_yeso_a_L1_por_L1() and not se_alimenta_clinker_a_L3():
#                     return "Húmeda a MC1 por MC1"
#         else:
#             print("No hay rutas disponibles para Puzolana Húmeda.")
#     # Ruta para Puzolana Seca
#     elif tipo_puzolana == "Puzolana_Seca":
#         if tiempo_vaciado_PS_L2 is not None or tiempo_vaciado_PS_L3 is not None:
#             if tiempo_vaciado_PS_L2 is not None and (tiempo_vaciado_PS_L3 is None or tiempo_vaciado_PS_L2 > tiempo_vaciado_PS_L3):
#                 if not se_alimenta_clinker_a_L3() and not se_alimenta_yeso_a_L3():
#                     ajustar_velocidad_transportador(1100, 1150)
#                     return "P.S a MC3 por MC2"
#             elif tiempo_vaciado_PS_L3 is not None and (tiempo_vaciado_PS_L2 is None or tiempo_vaciado_PS_L3 >= tiempo_vaciado_PS_L2):
#                 if not se_alimenta_clinker_a_L3() and not se_alimenta_yeso_a_L3():
#                     ajustar_velocidad_transportador(1100, 1150)
#                     return "P.S a 426HO02 por 426HO04"
#         else:
#             print("No hay rutas disponibles para Puzolana Seca.")

#     return "No se puede alimentar puzolana en este momento"


def seleccionar_ruta_puzolana(molino, tipo_puzolana, material_critico,material):
    print(f"\n--- Seleccionando ruta de Puzolana para {molino} ---")

    # Verificar el estado de las tolvas antes de seleccionar la ruta
    estado_tolvas = revisar_tolvas_y_evitar_alimentacion(molino)

    # Si la tolva de Puzolana está llena, no se selecciona una ruta de alimentación
    if estado_tolvas[molino].get(tipo_puzolana, False):
        print(f"Tolva de {tipo_puzolana} en {molino} está llena. No se seleccionará ruta de alimentación.")
        return f"No se puede alimentar más {tipo_puzolana} en {molino}, la tolva está llena --"


    # Calcular tiempos de vaciado solo si los molinos están en marcha
    tiempo_vaciado_PH_L1 = calcular_tiempo_vaciado("MC1", "Puzolana_Humeda") if verificar_estado_molino("MC1") else None
    tiempo_vaciado_PH_L2 = calcular_tiempo_vaciado("MC2", "Puzolana_Humeda") if verificar_estado_molino("MC2") else None
    tiempo_vaciado_PS_L2 = calcular_tiempo_vaciado("MC2", "Puzolana_Seca") if verificar_estado_molino("MC2") else None
    tiempo_vaciado_PS_L3 = calcular_tiempo_vaciado("MC3", "Puzolana_Seca") if verificar_estado_molino("MC3") else None

    # Verifica si los tiempos son None y ajusta el mensaje
    tiempo_vaciado_PH_L1_str = f"{tiempo_vaciado_PH_L1:.2f}h" if tiempo_vaciado_PH_L1 is not None else "Molino apagado"
    tiempo_vaciado_PH_L2_str = f"{tiempo_vaciado_PH_L2:.2f}h" if tiempo_vaciado_PH_L2 is not None else "Molino apagado"
    tiempo_vaciado_PS_L2_str = f"{tiempo_vaciado_PS_L2:.2f}h" if tiempo_vaciado_PS_L2 is not None else "Molino apagado"
    tiempo_vaciado_PS_L3_str = f"{tiempo_vaciado_PS_L3:.2f}h" if tiempo_vaciado_PS_L3 is not None else "Molino apagado"

    # Manejo de tiempos de vaciado negativos (verificación en las variables numéricas)
    if tiempo_vaciado_PH_L1 is not None and tiempo_vaciado_PH_L1 < 0:
        print("Advertencia: Tiempo de vaciado L1 es negativo, posiblemente el silo está vacío")
    if tiempo_vaciado_PH_L2 is not None and tiempo_vaciado_PH_L2 < 0:
        print("Advertencia: Tiempo de vaciado L2 es negativo, posiblemente el silo está vacío")
    if tiempo_vaciado_PS_L2 is not None and tiempo_vaciado_PS_L2 < 0:
        print("Advertencia: Tiempo de vaciado L2 es negativo, posiblemente el silo está vacío")
    if tiempo_vaciado_PS_L3 is not None and tiempo_vaciado_PS_L3 < 0:
        print("Advertencia: Tiempo de vaciado L3 es negativo, posiblemente el silo está vacío")

    print("alimentacion en progreso en MC1 ++++",verificar_alimentacion_simultanea(molino, material))
    # Priorizar Yeso en MC1 si el material crítico no es Puzolana Húmeda
    if molino == "MC1" and material_critico != "Puzolana_Humeda" and verificar_alimentacion_simultanea("MC1", "Yeso"):
        print("alimentacion en progreso en MC1",alimentaciones_en_progreso.get(molino, []))
        print(f"Material crítico en {molino} no es Puzolana Húmeda, priorizando alimentación de Yeso.")
        return "No se puede alimentar puzolana en este momento ,se está alimentando Yeso en MC1"

    # Ruta para Puzolana Húmeda
    if tipo_puzolana == "Puzolana_Humeda":
        print(f"Tiempos de vaciado: PH L1={tiempo_vaciado_PH_L1_str}, PH L2={tiempo_vaciado_PH_L2_str}")
        if tiempo_vaciado_PH_L1 is not None or tiempo_vaciado_PH_L2 is None:
            print("Alimentación a MC1")
            return "P.H a MC1 por MC1"

        if tiempo_vaciado_PH_L2 is not None or tiempo_vaciado_PH_L1 is None:
            print("Alimentación a MC1")
            return "P.H a MC2 por MC2"
        if tiempo_vaciado_PH_L1 is not None or tiempo_vaciado_PH_L2 is not None:
            if tiempo_vaciado_PH_L1 is not None and (tiempo_vaciado_PH_L2 is None or tiempo_vaciado_PH_L1 > tiempo_vaciado_PH_L2):
                if niveles_MC2["Puzolana_Humeda"] <= 9:
                    if se_requiere_alimentar_L2(niveles_MC2):
                        if not se_alimenta_yeso_a_L2_por_L2() and not se_alimenta_yeso_a_L3_por_L2() and not se_alimenta_puzolana_a_L1_por_L2():
                            return "P.H. a 426HO04 por MC2"
                    else:
                        if se_puede_alimentar_L1_por_L2():
                            return "Húmeda a MC1 por MC2"
            elif tiempo_vaciado_PH_L2 is not None and (tiempo_vaciado_PH_L1 is None or tiempo_vaciado_PH_L2 >= tiempo_vaciado_PH_L1):
                if not se_alimenta_yeso_a_L1_por_L1() and not se_alimenta_clinker_a_Silo_Blanco_L3():
                    return "Húmeda a MC1 por MC1"
        else:
            print("No hay rutas disponibles para Puzolana Húmeda.")

    # Ruta para Puzolana Seca
    elif tipo_puzolana == "Puzolana_Seca":
      if molino =="MC2":
        if tiempo_vaciado_PS_L2 is not None or tiempo_vaciado_PS_L3 is not None:
            if tiempo_vaciado_PS_L2 is not None and (tiempo_vaciado_PS_L3 is None or tiempo_vaciado_PS_L2 > tiempo_vaciado_PS_L3):
                if not se_alimenta_clinker_a_Silo_Blanco_L3() and not se_alimenta_yeso_a_L3():
                    ajustar_velocidad_transportador(1100, 1150)
                    #return "P.S a MC3 por MC2"
                    return "P.S a 426HO02 por 426HO04"
        else:
            print("No hay rutas disponibles para Puzolana Seca en MC2.")

      if molino == "MC3":
         print("Buscando ruta para Puzolana Seca en MC3 ...")
         print("tiempo de vaciado PS L3",tiempo_vaciado_PS_L3)
         if tiempo_vaciado_PS_L3 is not None and (tiempo_vaciado_PS_L2 is not None or tiempo_vaciado_PS_L3 >= tiempo_vaciado_PS_L2):
            if not se_alimenta_clinker_a_Silo_Blanco_L3(): #and not se_alimenta_yeso_a_L3():
                  ajustar_velocidad_transportador(1100, 1150)
                  #return "P.S a 426HO02 por 426HO04"
                  return "P.S a MC3 por MC2"
            else:
                print("No hay rutas disponibles para Puzolana Seca en MC3.")

    return "No se puede alimentar puzolana en este momento ´´"




def seleccionar_ruta_yeso(molino,material_critico):
    print(f"\n--- Seleccionando ruta para Yeso en {molino} ---")

        # Verificar el estado de las tolvas antes de seleccionar la ruta
    estado_tolvas = revisar_tolvas_y_evitar_alimentacion(molino)

    # Si la tolva de Puzolana está llena, no se selecciona una ruta de alimentación
    if estado_tolvas[molino].get("Yeso", False):
        print(f"Tolva de Yeso en {molino} está llena. No se seleccionará ruta de alimentación.")
        return f"No se puede alimentar más Yeso en {molino}, la tolva está llena --"

    # Calcular tiempos de vaciado solo para molinos que están en marcha
    tiempo_vaciado_L1 = calcular_tiempo_vaciado("MC1", "Yeso") if verificar_estado_molino("MC1") else None
    tiempo_vaciado_L2 = calcular_tiempo_vaciado("MC2", "Yeso") if verificar_estado_molino("MC2") else None
    tiempo_vaciado_L3 = calcular_tiempo_vaciado("MC3", "Yeso") if verificar_estado_molino("MC3") else None

    # Formatear los tiempos para su impresión
    tiempo_vaciado_L1_str = f"{tiempo_vaciado_L1:.2f}h" if tiempo_vaciado_L1 is not None else "Molino apagado"
    tiempo_vaciado_L2_str = f"{tiempo_vaciado_L2:.2f}h" if tiempo_vaciado_L2 is not None else "Molino apagado"
    tiempo_vaciado_L3_str = f"{tiempo_vaciado_L3:.2f}h" if tiempo_vaciado_L3 is not None else "Molino apagado"

    print(f"Tiempos de vaciado: L1={tiempo_vaciado_L1_str}, L2={tiempo_vaciado_L2_str}, L3={tiempo_vaciado_L3_str}")

    # Priorizar Yeso en MC1 si el material crítico no es Puzolana Húmeda
    if molino == "MC1" and material_critico != "Yeso":
        print(f"Material crítico en {molino} no es Yeso, priorizando alimentación de Puzola Húmeda.")
        return "No se puede alimentar yeso en este momento"


    if molino == "MC1":
        print("Verificando condiciones para MC1...")
        if se_puede_alimentar_por_L1():
            print("Se puede alimentar por L1")
            if puede_alimentar_yeso_a_L1():
                print("Se puede alimentar Yeso a L1")
                if not se_alimenta_puzolana_a_L1():
                    print("No se está alimentando Puzolana a L1")
                    if tiempo_vaciado_L1 is not None and tiempo_vaciado_L2 is not None:
                        if tiempo_vaciado_L1 > tiempo_vaciado_L2:
                            print("Tiempo de vaciado en L1 es mayor que en L2")
                            return "Hacia MC1 por MC2"
                        else:
                            print("Alimentando Yeso a L1 por L1")
                            return "Hacia MC1 por MC1"
                    else:
                        if not se_alimenta_puzolana_a_L1():
                            return "Hacia MC1 por MC1"
                else:
                    print("Puzolana siendo alimentada a L1. Yeso en espera.")
                    return "En espera para MC1 por MC1"
            else:
                print("No se puede alimentar Yeso a L1")
        elif se_puede_alimentar_L1_por_L2():
            print("Se puede alimentar L1 por L2")
            if not se_alimenta_yeso_a_L2_por_L2():
             print("No se está alimentando Yeso a L2")
             if not se_alimenta_yeso_a_L3_por_L2():
              print("No se está alimentando Yeso a L3")
              if not se_alimenta_yeso_a_L1_por_L2():
                print("No se está alimentando Yeso a L1 por L2")
                if not se_alimenta_puzolana_a_L1_por_L2():
                  print("No se está alimentando Puzolana a L1 por L2")
                  if not se_alimenta_PH_a_426HO04_por_L2():
                    print("No se está alimentando Puzolana húmeda a 426HO04 por L2")
                    return "Hacia MC1 por MC2"
                  else:
                    print("Se está alimentando Puzolana, no se puede alimentar Yeso por L2")
                    return "No se puede alimentar Yeso en este momento 1"

    elif molino == "MC2":
        print("Verificando condiciones para MC2...")
        if not se_alimenta_yeso_a_L2_por_L2():
          print("No se está alimentando Yeso a L2")
          if not se_alimenta_yeso_a_L3_por_L2():
            print("No se está alimentando Yeso a L3")

            if tiempo_vaciado_L2 is None or tiempo_vaciado_L2 < 0:
              print("Alimentación a MC2 desde pretrit")
              return "Yeso Hacia MC2 por MC2"

            if tiempo_vaciado_L1 is not None and tiempo_vaciado_L2 is not None:
                if tiempo_vaciado_L2 > tiempo_vaciado_L1:
                    print("Tiempo de vaciado en L2 es mayor que en L1")
                    return "Hacia M1 por MC1 "
                else:
                    print("Alimentando Yeso a MC2 por L2")
                    return "Hacia MC2 por MC2"

            if tiempo_vaciado_L2 is not None and tiempo_vaciado_L3 is not None:
                if tiempo_vaciado_L2 > tiempo_vaciado_L3:
                    print("Tiempo de vaciado en L2 es mayor que en L3")
                    return "Hacia MC2 por L3"
                else:
                    print("Alimentando Yeso a MC2 por L2")
                    return "Hacia MC2 por MC2"
            else:
                print("No se puede calcular tiempos de vaciado adecuados para MC2")
                return "No se puede alimentar Yeso en este momento 2"
        else:
            print("Se está alimentando Puzolana, no se puede alimentar Yeso")
            return "No se puede alimentar Yeso en este momento en MC2"

    elif molino == "MC3":
        print("Verificando condiciones para MC3...")
        if se_puede_alimentar_por_L1():
            print("Se puede alimentar por L1")
            if not se_alimenta_clinker_a_L3():
                print("No se está alimentando Clinker a L3")
                if tiempo_vaciado_L1 is not None and tiempo_vaciado_L2 is not None:
                    if tiempo_vaciado_L1 > tiempo_vaciado_L2:
                        print("Alimentando Yeso a MC3 por L2")
                        return "Hacia MC3 por L2"
                    else:
                        print("Alimentando Yeso a MC3 por L1")
                        return "Hacia MC3 por L1"
                else:
                    print("No se puede calcular tiempos de vaciado adecuados para MC3")
                    return "No se puede alimentar Yeso en este momento en MC3 - 1"
            else:
                print("Se está alimentando Clinker a L3, no se puede alimentar Yeso por L1")
                return "No se puede alimentar Yeso en este momento en MC3 - 2"
        elif not se_alimenta_yeso_a_L2_por_L2():
             if not se_alimenta_yeso_a_L1_por_L2():
              print("No se está alimentando Yeso a L1 por L2")
              if not se_alimenta_puzolana_a_L1_por_L2():
                print("No se está alimentando Puzolana a L1 por L2")
                if not se_alimenta_PH_a_426HO04_por_L2():
                  print("No se está alimentando Puzolana húmeda a 426HO04 por L2")
                  return "Hacia MC3 por MC2"
                else:
                  print("Se está alimentando Puzolana, no se puede alimentar Yeso por L2")
                  return "No se puede alimentar Yeso en este momento en MC3 - 3"

    print("No se encontró una ruta válida para alimentar Yeso")
    return "No se puede alimentar Yeso en este momento, posiblemente la tolva está llena"



# def esperar_y_alimentar(molino, material, cantidad):
#     while se_alimenta_puzolana_a_L1():
#         time.sleep(1)  # Espera 1 segundo antes de volver a verificar

#     print(f"Intentando nuevamente alimentar {material} a {molino}")
#     intentar_alimentar(molino, material, cantidad)


def validar_restricciones(molino, material):
    with lock:
        # # Verificar si el material ya está siendo alimentado en este molino
        # if molino in alimentaciones_en_progreso and material in alimentaciones_en_progreso[molino]:
        #     return False, f"Ya se está alimentando {material} a {molino}."


        # Validación para MC1
        if molino == "MC1":
            # Permitir siempre la alimentación de Clinker
            if material == "Clinker":
                print("clinkereeeeeeeerr")
                return True, "Validación exitosa"

            # Restricción: No permitir alimentar simultáneamente Yeso y Puzolana
            if material == "Yeso":
                print("yesooooooooo")
                if "Puzolana_Humeda" in alimentaciones_en_progreso.get(molino, []):
                    print("alimentaciones_en_progreso ****",alimentaciones_en_progreso.get(molino, []))
                    return False, "No se puede alimentar Yeso y Puzolana simultáneamente en MC1."
                # Restricción adicional: Límite de 2 metros para Yeso en MC1
                if niveles_MC1["Yeso"] + 2 > tolvas_MC1["Yeso"]["max_metros"]:
                    return False, "No se puede alimentar más Yeso a MC1, se excedería el límite de 2m."
            if material == "Puzolana_Humeda":
                print("puzolnaaaaaaaaaaaaaaa")
                if "Yeso" in alimentaciones_en_progreso.get(molino, []):
                    return False, "No se puede alimentar Puzolana y Yeso simultáneamente en MC1."

            # Permitir alimentar Puzolana_Humeda o Yeso cuando no estén en conflicto
            return True, "Validación exitosa"

        # Validación para MC2
        if molino == "MC2":
            # No permitir alimentar Puzolana Húmeda y Puzolana Seca al mismo tiempo
            if material == "Puzolana_Humeda" and "Puzolana_Seca" in alimentaciones_en_progreso.get(molino, []):
                return False, "No se puede alimentar Puzolana Húmeda y Seca simultáneamente en MC2."
            if material == "Puzolana_Seca" and "Puzolana_Humeda" in alimentaciones_en_progreso.get(molino, []):
                return False, "No se puede alimentar Puzolana Seca y Húmeda simultáneamente en MC2."

            # No permitir alimentar Yeso y Puzolana simultáneamente
            if material == "Yeso":
                if "Puzolana_Humeda" in alimentaciones_en_progreso.get(molino, []) or "Puzolana_Seca" in alimentaciones_en_progreso.get(molino, []):
                    return False, "No se puede alimentar Yeso y Puzolana simultáneamente en MC2."
                # Restricción adicional: Evitar sobrealimentación de Yeso en MC2
                if niveles_MC2["Yeso"] > tolvas_MC2["Yeso"]["max_metros"]:
                    return False, "No se puede alimentar más Yeso a MC2, se excedería el límite."

            # Permitir alimentar Puzolana o Yeso cuando no estén en conflicto
            return True, "Validación exitosa"

        # Validación para MC3
        if molino == "MC3":
            # No permitir alimentar simultáneamente Clinker, Puzolana y Yeso
            if material == "Clinker_Silo_Blanco":
                # Verificar que no se esté alimentando Puzolana o Yeso al mismo tiempo
                if any(mat in ["Puzolana_Seca", "Yeso"] for mat in alimentaciones_en_progreso.get(molino, [])):
                    return False, "No se puede alimentar Clinker, Puzolana y Yeso simultáneamente en MC3."
            if material == "Puzolana_Seca":
                if "Clinker_Silo_Blanco" in alimentaciones_en_progreso.get(molino, []):
                    return False, "No se puede alimentar Puzolana Seca y Clinker simultáneamente en MC3."
                if "Yeso" in alimentaciones_en_progreso.get(molino, []):
                    return False, "No se puede alimentar Puzolana y Yeso simultáneamente en MC3."
            if material == "Yeso":
                if "Puzolana_Seca" in alimentaciones_en_progreso.get(molino, []):
                    return False, "No se puede alimentar Yeso y Puzolana simultáneamente en MC3."
                if "Clinker_Silo_Blanco" in alimentaciones_en_progreso.get(molino, []):
                    return False, "No se puede alimentar Yeso y Clinker simultáneamente en MC3."

            # Permitir la alimentación si no hay conflictos
            return True, "Validación exitosa"

    return True, "Validación exitosa"



def intentar_alimentar(molino, material, cantidad,material_critico):
    #print(f"\n--- Intentando alimentar {material} a {molino} ---")
    #print(f"Cantidad solicitada: {cantidad:.2f}t")

    # Verificar si ya se está alimentando este material en este molino
    with lock:
        if molino in alimentaciones_en_progreso and material in alimentaciones_en_progreso[molino]:
            print(f"Ya se está alimentando {material} a {molino}. No se puede iniciar otra alimentación simultánea.")
            return 0

    # Verificar nuevamente si es necesario alimentar
    if molino == "MC1":
        niveles = niveles_MC1_toneladas
        tolvas = tolvas_MC1
        rendimiento=0.8
    elif molino == "MC2":
        niveles = niveles_MC2_toneladas
        tolvas = tolvas_MC2
        rendimiento=0.8
    else:  # MC3
        niveles = niveles_MC3_toneladas
        tolvas = tolvas_MC3
        rendimiento=0.5
    nivel_actual = niveles[material]
    tipo_produccion = tipos_produccion_actual[molino]

    if tipo_produccion is None:
        print(f"No se ha definido un tipo de producción para {molino}. No se puede alimentar {material}.")
        return 0

    necesidades = calcular_necesidades(molino, tipo_produccion)
    cantidad_necesaria = necesidades[material]

    if nivel_actual >= cantidad_necesaria:
        print(f"Ya no es necesario alimentar {material} en {molino}. Nivel actual: {nivel_actual:.2f}t, Necesario: {cantidad_necesaria:.2f}t")
        return 0

    ruta = seleccionar_ruta_alimentacion(molino, material,material_critico)
    print(f"Ruta seleccionada: {ruta}")

    # if ruta == "En espera para MC1 por MC1":
    #     print(f"Alimentación de {material} a {molino} en espera. Se intentará nuevamente cuando termine la alimentación de Puzolana.")
    #     threading.Thread(target=esperar_y_alimentar, args=(molino, material, cantidad)).start()
    #     return 0

    if ruta == "No se puede alimentar Yeso en este momento" or ruta == "Material no reconocido":
        print(f"Error: No se puede alimentar {material} a {molino}: {ruta}")
        return 0

    # Verificar restricciones
    print("Verificando restricciones...")
    validacion, mensaje = validar_restricciones(molino, material)
    if not validacion:
        print(f"No se puede alimentar debido a restricciones: {mensaje}")
        return 0



    with lock:
        print("Agregando alimentación a las listas de control...")
        alimentaciones_actuales.add((molino, material))  # Cambiado de .append() a .add()
        if molino not in alimentaciones_en_progreso:
            alimentaciones_en_progreso[molino] = set()
        alimentaciones_en_progreso[molino].add(material)

    print(f"Ajustando velocidad del transportador para {material}...")
    if material in ["Puzolana", "Puzolana_Humeda", "Puzolana_Seca"]:
        velocidad = ajustar_velocidad_transportador(1100, 1150)
    elif material == "Yeso":
        velocidad = ajustar_velocidad_transportador(850, 900)
    else:
        velocidad = "No ajustada"
    print(f"Velocidad del transportador: {velocidad}")

    capacidad = tolvas[material]['capacidad']
    print(f"Capacidad de la tolva: {capacidad}t")

    if 'max_metros' in tolvas[material]:
        max_nivel = tolvas[material]['max_metros']*rendimiento
        nivel_actual = niveles[material] / capacidad * max_nivel
        #print(f"Nivel actual: {nivel_actual:.2f}m / {max_nivel}m")
        if molino == "MC1" and material == "Yeso":
            max_nivel = min(max_nivel, 2)  # Limitar a 2m para yeso en MC1
           # print(f"Nivel máximo ajustado para Yeso en MC1: {max_nivel}m")
        espacio_disponible = max_nivel - nivel_actual
        cantidad_metros = min(cantidad / capacidad * max_nivel, espacio_disponible)
        cantidad_alimentada = cantidad_metros / max_nivel * capacidad
        #print(f"Espacio disponible: {espacio_disponible:.2f}m")
        #print(f"Cantidad a alimentar en metros: {cantidad_metros:.2f}m")
    else:  # Para MC3 que usa porcentajes
        max_nivel = tolvas[material]['max_porcentaje']*rendimiento
        nivel_actual = niveles[material] / capacidad * 100
        #print(f"Nivel actual: {nivel_actual:.2f}% / {max_nivel}%")
        espacio_disponible = max_nivel - nivel_actual
        cantidad_porcentaje = min(cantidad / capacidad * 100, espacio_disponible)
        cantidad_alimentada = cantidad_porcentaje / 100 * capacidad
        #print(f"Espacio disponible: {espacio_disponible:.2f}%")
        #print(f"Cantidad a alimentar en porcentaje: {cantidad_porcentaje:.2f}%")

    #print(f"Cantidad final a alimentar: {cantidad_alimentada:.2f}t")

    if cantidad_alimentada > 0:
        #print(f"Iniciando alimentación de {cantidad_alimentada:.2f}t de {material} a {molino}")
        iniciar_alimentacion(molino, material, cantidad_alimentada)

    else:
        print(f"No se pudo alimentar {material} a {molino}. Tolva llena o se alcanzó el límite máximo.")
        with lock:
            liberar_rutas_alimentacion(molino, material)
            alimentaciones_actuales.discard((molino, material))  # Cambiado de .remove() a .discard()
            if molino in alimentaciones_en_progreso and material in alimentaciones_en_progreso[molino]:
                alimentaciones_en_progreso[molino].discard(material)  # Cambiado de .remove() a .discard()

    return cantidad_alimentada

def iniciar_alimentacion(molino, material, cantidad):
    if molino not in alimentaciones_en_progreso:
        alimentaciones_en_progreso[molino] = set()
    alimentaciones_en_progreso[molino].add(material)
    #temporizadores_molinos[molino] = True
    #thread = threading.Thread(target=simular_alimentacion, args=(molino, material, cantidad, 13))    # Aquí se coloca la relación de la simulación de alimentación de horas a segundos
    #thread.start()
    print(f"\nIniciando alimentación de {material} a {molino}")



def simular_alimentacion(molino, material, cantidad, duracion_real):
    tiempo_inicio = time.time()
    tiempo_simulado = 0

    while tiempo_simulado < 60:  # Simular 1 hora
        tiempo_actual = time.time() - tiempo_inicio
        tiempo_simulado = (tiempo_actual / duracion_real) * 60  # Calcular tiempo simulado
        porcentaje = min(tiempo_simulado / 60 * 100, 100)  # Calcular porcentaje de alimentación
        #imprimir_progreso(molino, material, porcentaje)
        time.sleep(0.1)  # Simulación en tiempo real

    with lock_print:
        print(f"\n{molino} - Alimentación de {material} completada.")

    # Calcular el nivel necesario antes de actualizar
    tipo_produccion = tipos_produccion_actual.get(molino)
    necesidades = calcular_necesidades(molino, tipo_produccion)
    necesario = necesidades.get(material, 0)

    # Actualizar niveles después de la alimentación solo si es necesario
    with lock:
        if molino == "MC1":
            nivel_actual = niveles_MC1_toneladas.get(material, 0)
            if nivel_actual < necesario:
                cantidad_a_alimentar = min(cantidad, necesario - nivel_actual)
                niveles_MC1_toneladas[material] += cantidad_a_alimentar  # Actualizar el nivel en MC1
            else:
                print(f"Nivel suficiente para {material} en MC1. No se alimentará más.")
        elif molino == "MC2":
            nivel_actual = niveles_MC2_toneladas.get(material, 0)
            if nivel_actual < necesario:
                cantidad_a_alimentar = min(cantidad, necesario - nivel_actual)
                niveles_MC2_toneladas[material] += cantidad_a_alimentar  # Actualizar el nivel en MC2
            else:
                print(f"Nivel suficiente para {material} en MC2. No se alimentará más.")
        elif molino == "MC3":
            nivel_actual = niveles_MC3_toneladas.get(material, 0)
            if nivel_actual < necesario:
                cantidad_a_alimentar = min(cantidad, necesario - nivel_actual)
                niveles_MC3_toneladas[material] += cantidad_a_alimentar  # Actualizar el nivel en MC3
            else:
                print(f"Nivel suficiente para {material} en MC3. No se alimentará más.")

        # Imprimir el estado actual de los niveles después de la alimentación
        print(f"\nEstado actual de las tolvas después de la alimentación:")
        print(f"Niveles MC1: {redondear_diccionario(niveles_MC1_toneladas)}")
        print(f"Niveles MC2: {redondear_diccionario(niveles_MC2_toneladas)}")
        print(f"Niveles MC3: {redondear_diccionario(niveles_MC3_toneladas)}")

    #Actualizar listas de alimentaciones en progreso
    with lock:
        if material in alimentaciones_en_progreso.get(molino, []):
            print(f"Eliminando {material} de alimentaciones en progreso para {molino}")
            alimentaciones_en_progreso[molino].remove(material)

        if (molino, material) in alimentaciones_actuales:
            print(f"Eliminando {(molino, material)} de alimentaciones actuales")
            alimentaciones_actuales.remove((molino, material))

        print(f"\n{molino} - Alimentación de {material} completada. Ruta habilitada para nueva alimentación.")

    #1Actualizar temporizador
    if not alimentaciones_en_progreso.get(molino):
        temporizadores_molinos[molino] = False




    # Actualizar listas de alimentaciones en progreso (con verificaciones adicionales)
    with lock:
        if material in alimentaciones_en_progreso.get(molino, []):
            print(f"Eliminando {material} de alimentaciones en progreso para {molino}")
            alimentaciones_en_progreso[molino].remove(material)

        if (molino, material) in alimentaciones_actuales:
            print(f"Eliminando {(molino, material)} de alimentaciones actuales")
            alimentaciones_actuales.remove((molino, material))

        print(f"\n{molino} - Alimentación de {material} completada. Ruta habilitada para nueva alimentación.")

    # # Actualizar temporizador
    # if not alimentaciones_en_progreso.get(molino):
    #     temporizadores_molinos[molino] = False

def simular_alimentacion(molino, material, cantidad, duracion_real):
    # tiempo_inicio = time.time()
    # tiempo_simulado = 0

    # while tiempo_simulado < 60:  # Simular 1 hora
    #     tiempo_actual = time.time() - tiempo_inicio
    #     tiempo_simulado = (tiempo_actual / duracion_real) * 60  # Calcular tiempo simulado
    #     porcentaje = min(tiempo_simulado / 60 * 100, 100)  # Calcular porcentaje de alimentación
    #     imprimir_progreso(molino, material, porcentaje)
    #     time.sleep(0.1)  # Simulación en tiempo real

    # with lock_print:
    #     print(f"\n{molino} - Alimentación de {material} completada.")

    # # Actualizar niveles después de la alimentación
    # with lock:
    #     if molino == "MC1":
    #         if material in niveles_MC1_toneladas:
    #             print("niveles_MC1_toneladas ", niveles_MC1_toneladas[material])
    #             print("cantidad_a_alimentar", cantidad)
    #             niveles_MC1_toneladas[material] += cantidad  # Actualizar el nivel en MC1
    #         else:
    #             print(f"Advertencia: No se encontró el material '{material}' en MC1. No se realizó la alimentación.")

    #     elif molino == "MC2":
    #         if material in niveles_MC2_toneladas:
    #             niveles_MC2_toneladas[material] += cantidad  # Actualizar el nivel en MC2
    #         else:
    #             print(f"Advertencia: No se encontró el material '{material}' en MC2. No se realizó la alimentación.")

    #     elif molino == "MC3":
    #         if material in niveles_MC3_toneladas:
    #             niveles_MC3_toneladas[material] += cantidad  # Actualizar el nivel en MC3
    #         else:
    #             print(f"Advertencia: No se encontró el material '{material}' en MC3. No se realizó la alimentación.")

    #     # Imprimir el estado actual de los niveles después de la alimentación
    #     print(f"\nEstado actual de las tolvas después de la alimentación:")
    #     print(f"Niveles MC1: {redondear_diccionario(niveles_MC1_toneladas)}")
    #     print(f"Niveles MC2: {redondear_diccionario(niveles_MC2_toneladas)}")
    #     print(f"Niveles MC3: {redondear_diccionario(niveles_MC3_toneladas)}")

    # Actualizar listas de alimentaciones en progreso (sin volver a imprimir los niveles)
    with lock:
        if material in alimentaciones_en_progreso.get(molino, []):
            print(f"Eliminando {material} de alimentaciones en progreso para {molino}")
            alimentaciones_en_progreso[molino].remove(material)

        if (molino, material) in alimentaciones_actuales:
            print(f"Eliminando {(molino, material)} de alimentaciones actuales")
            alimentaciones_actuales.remove((molino, material))

        print(f"\n{molino} - Alimentación de {material} completada. Ruta habilitada para nueva alimentación.")

    # Actualizar temporizador
    if not alimentaciones_en_progreso.get(molino):
        temporizadores_molinos[molino] = False


def verificar_alimentacion_simultanea(molino, material):
    if molino == "MC1":
        # Verificar si se está alimentando Yeso o Puzolana simultáneamente
        if "Yeso" in alimentaciones_en_progreso.get("MC1", []) and material == "Puzolana_Humeda":
            print(f"No se puede alimentar {material} en MC1, ya se está alimentando Yeso.")
            return False
        elif "Puzolana_Humeda" in alimentaciones_en_progreso.get("MC1", []) and material == "Yeso":
            print(f"No se puede alimentar Yeso en MC1, ya se está alimentando Puzolana Húmeda.")
            return False
    return True

def finalizar_alimentacion(molino, material):
    with lock:
        if material in alimentaciones_en_progreso.get(molino, []):
            alimentaciones_en_progreso[molino].remove(material)
            print(f"Eliminando {material} de alimentaciones en progreso para {molino}.")
        else:
            print(f"{material} no estaba en progreso en {molino}.")




# def optimizar_alimentacion(molino):
#     if molino == "MC1":
#         niveles = niveles_MC1_toneladas
#     elif molino == "MC2":
#         niveles = niveles_MC2_toneladas
#     else:  # MC3
#         niveles = niveles_MC3_toneladas

#     # Verificar tiempos de vaciado para cada material
#     tiempos_vaciado = {}
#     for material in niveles:
#         tiempo = calcular_tiempo_vaciado(molino, material)
#         if tiempo != float('inf'):  # Considerar solo si el consumo es válido
#             tiempos_vaciado[material] = tiempo

#     # Ordenar los materiales por el tiempo de vaciado más cercano a agotarse
#     tiempos_vaciado = {k: v for k, v in sorted(tiempos_vaciado.items(), key=lambda item: item[1])}

#     print(f"\nTiempos de vaciado para {molino}:")
#     for material, tiempo in tiempos_vaciado.items():
#         print(f"  {material}: {tiempo:.2f} horas para vaciado")

#     # Verificar si hay materiales críticos (tiempo de vaciado negativo)
#     material_critico = next(iter(tiempos_vaciado))
#     if tiempos_vaciado[material_critico] < 0:
#         print(f"Material crítico en {molino}: {material_critico}. Priorizar su alimentación.")
#         intentar_alimentar(molino, material_critico, calcular_necesidades(molino, tipos_produccion_actual[molino]).get(material_critico, 0),material_critico)
#         return

#     # Continuar alimentando el material más cercano a agotarse
#     material_a_alimentar = next(iter(tiempos_vaciado))
#     print(f"Material más crítico en {molino} para alimentar: {material_a_alimentar}")
#     cantidad_a_alimentar = calcular_necesidades(molino, tipos_produccion_actual[molino]).get(material_a_alimentar, 0)

#     # if cantidad_a_alimentar > 0:
#     #     intentar_alimentar(molino, material_a_alimentar, cantidad_a_alimentar)
#     # else:
#     #     print(f"No es necesario alimentar {material_a_alimentar} en este ciclo.")

#     if cantidad_a_alimentar > 0:
#         intentar_alimentar(molino, material_a_alimentar, cantidad_a_alimentar, material_critico)
#     else:
#         print(f"No es necesario alimentar {material_a_alimentar} en este ciclo.")

# def imprimir_tiempos(molino, tipo_produccion):
#     necesidades = calcular_necesidades(molino, tipo_produccion)
#     tiempos_llenado = calcular_tiempos_llenado(molino, necesidades)

#     # print(f"\nTiempos de llenado y vaciado para {molino} - {tipo_produccion}:")
#     # print("-" * 60)
#     # print(f"{'Material':<15}{'Tiempo Llenado (h)':<20}{'Tiempo Vaciado (h)':<20}")
#     # print("-" * 60)

#     print(f"\nTiempos de vaciado para {molino} - {tipo_produccion}:")
#     print("-" * 60)
#     print(f"{'Material':<23}{'Tiempo Vaciado (h)':<20}")
#     print("-" * 60)

#     for material, tiempo_llenado in tiempos_llenado.items():
#         tiempo_vaciado = calcular_tiempo_vaciado(molino, material)
#         tiempo_llenado_str = f"{tiempo_llenado:.2f}" if tiempo_llenado != float('inf') else "∞"
#         tiempo_vaciado_str = f"{tiempo_vaciado:.2f}" if tiempo_vaciado != float('inf') else "∞"
#         #print(f"{material:<15}{tiempo_llenado_str:<20}{tiempo_vaciado_str:<20}")
#         print(f"{material:<23}{tiempo_vaciado_str:<20}")

def optimizar_alimentacion(molino):
    # Obtener los niveles de toneladas según el molino
    if molino == "MC1":
        niveles = niveles_MC1_toneladas
    elif molino == "MC2":
        niveles = niveles_MC2_toneladas
    else:  # MC3
        niveles = niveles_MC3_toneladas

    # Verificar tiempos de vaciado para cada material
    tiempos_vaciado = {}
    for material in niveles:
        tiempo = calcular_tiempo_vaciado(molino, material)
        if tiempo != float('inf'):  # Considerar solo si el consumo es válido
            tiempos_vaciado[material] = tiempo

    # Ordenar los materiales por el tiempo de vaciado más cercano a agotarse
    tiempos_vaciado = {k: v for k, v in sorted(tiempos_vaciado.items(), key=lambda item: item[1])}

    print(f"\nTiempos de vaciado para {molino}:")
    for material, tiempo in tiempos_vaciado.items():
        print(f"  {material}: {tiempo:.2f} horas para vaciado")

    # Verificar si hay materiales críticos (tiempo de vaciado negativo)
    material_critico = next(iter(tiempos_vaciado))
    if tiempos_vaciado[material_critico] < 0:
        print(f"Material crítico en {molino}: {material_critico}. Priorizar su alimentación.")
        # Verificar si la tolva correspondiente está llena antes de alimentar
        estado_tolvas = revisar_tolvas_y_evitar_alimentacion(molino)
        if not estado_tolvas[molino][material_critico]:
            intentar_alimentar(molino, material_critico, calcular_necesidades(molino, tipos_produccion_actual[molino]).get(material_critico, 0), material_critico)
        else:
            print(f"No se puede alimentar más {material_critico} en {molino}, la tolva está llena.")
        return

    # Continuar alimentando el material más cercano a agotarse
    material_a_alimentar = next(iter(tiempos_vaciado))
    print(f"Material más crítico en {molino} para alimentar: {material_a_alimentar}")
    cantidad_a_alimentar = calcular_necesidades(molino, tipos_produccion_actual[molino]).get(material_a_alimentar, 0)

    # Verificar si la tolva correspondiente está llena antes de alimentar
    estado_tolvas = revisar_tolvas_y_evitar_alimentacion(molino)
    if not estado_tolvas[molino][material_a_alimentar]:
        if cantidad_a_alimentar > 0:
            intentar_alimentar(molino, material_a_alimentar, cantidad_a_alimentar, material_critico)
        else:
            print(f"No es necesario alimentar {material_a_alimentar} en este ciclo.")
    else:
        print(f"No se puede alimentar más {material_a_alimentar} en {molino}, la tolva está llena.")



def generar_recomendaciones(molino):   # actualizado 15/10/24
    recomendaciones = []
    restricciones = []

    # Obtener tiempos de vaciado
    tiempos_vaciado = {}
    for material in niveles_MC1_toneladas if molino == "MC1" else niveles_MC2_toneladas if molino == "MC2" else niveles_MC3_toneladas:
        # Verificar si el material es Puzolana Húmeda o Seca y si corresponde al molino
        if (molino == "MC1" and material == "Puzolana_Humeda") or \
           (molino == "MC2" and material in ["Puzolana_Humeda", "Puzolana_Seca"]) or \
           (molino == "MC3" and material == "Puzolana_Seca"):
            tiempo_vaciado = calcular_tiempo_vaciado(molino, material)
            if tiempo_vaciado != float('inf'):  # Solo recomendar si el consumo es válido
                tiempos_vaciado[material] = tiempo_vaciado
        else:
            # Para otros materiales que no son puzolanas
            tiempo_vaciado = calcular_tiempo_vaciado(molino, material)
            if tiempo_vaciado != float('inf'):
                tiempos_vaciado[material] = tiempo_vaciado

    # Ordenar los materiales por tiempos de vaciado
    tiempos_vaciado = sorted(tiempos_vaciado.items(), key=lambda item: item[1])

    for material, tiempo in tiempos_vaciado:
        # if tiempo <= 2:  # Si el tiempo de vaciado es menor o igual a 2 horas

        #     recomendaciones.append(f"Recomiendo alimentar {material} en {molino}, quedan {tiempo:.2f} horas antes del vaciado.")
        # else:
        #     restricciones.append(f"El tiempo de vaciado de {material} en {molino} es de {tiempo:.2f} horas. No es urgente alimentarlo ahora.")
          recomendaciones.append(f"Recomiendo alimentar {material} en {molino}, quedan {tiempo:.2f} horas antes del vaciado.")
    if not recomendaciones:
        recomendaciones.append(f"Todas las tolvas de {molino} tienen suficiente material en este momento.")

    return recomendaciones, restricciones

# def generar_recomendaciones_para_todos_los_molinos():
#     recomendaciones_generales = []
#     restricciones_generales = []
#     tiempos_comparativos = {}

#     for molino in ["MC1", "MC2", "MC3"]:
#         # Verificar si el molino está apagado
#         if not verificar_estado_molino(molino):
#             print(f"El molino {molino} está apagado. No se generarán recomendaciones para este molino.")
#             continue  # Saltar molino apagado

#         # Obtener tiempos de vaciado para cada material
#         tiempos_vaciado = {}

#         for material in niveles_MC1_toneladas if molino == "MC1" else niveles_MC2_toneladas if molino == "MC2" else niveles_MC3_toneladas:
#             # Incluir solo los materiales correspondientes a cada molino
#             if (molino == "MC1" and material == "Puzolana_Humeda") or \
#                (molino == "MC2" and material in ["Puzolana_Humeda", "Puzolana_Seca"]) or \
#                (molino == "MC3" and material == "Puzolana_Seca"):
#                 tiempo_vaciado = calcular_tiempo_vaciado(molino, material)
#                 if tiempo_vaciado != float('inf'):
#                     tiempos_vaciado[material] = tiempo_vaciado
#             else:
#                 # Para otros materiales
#                 tiempo_vaciado = calcular_tiempo_vaciado(molino, material)
#                 if tiempo_vaciado != float('inf'):
#                     tiempos_vaciado[material] = tiempo_vaciado

#         # Guardar los tiempos para comparaciones entre molinos
#         for material, tiempo in tiempos_vaciado.items():
#             if material not in tiempos_comparativos:
#                 tiempos_comparativos[material] = {}
#             tiempos_comparativos[material][molino] = tiempo

#         # Generar recomendaciones para cada molino
#         recomendaciones, restricciones = generar_recomendaciones(molino)
#         recomendaciones_generales.extend(recomendaciones)
#         restricciones_generales.extend(restricciones)

#     # Comparar tiempos de vaciado entre molinos y seleccionar la mejor ruta de alimentación
#     print("\n--- Comparación de tiempos de vaciado entre molinos ---")

#     for material, tiempos in tiempos_comparativos.items():
#         print(f"\nMaterial: {material}")
#         for molino, tiempo in tiempos.items():
#             if tiempo is not None:  # Verifica si el tiempo no es None
#                 print(f"  {molino}: {tiempo:.2f} horas para vaciado")
#             else:
#                 print(f"  {molino}: Molino apagado")

#         # Seleccionar la ruta más crítica (el molino que necesita ser alimentado primero)
#         molino_mas_critico = min(tiempos, key=tiempos.get)
#         print(f"\nRuta más crítica para {material}: {molino_mas_critico}")

#         # Recomendar la alimentación a la ruta más crítica
#         ruta = seleccionar_ruta_alimentacion(molino_mas_critico, material)
#         recomendaciones_generales.append(f"Recomiendo alimentar {material} hacia {molino_mas_critico} usando la ruta: {ruta}")

#     return recomendaciones_generales, restricciones_generales


# def generar_recomendaciones_para_todos_los_molinos():
#     recomendaciones_generales = []
#     restricciones_generales = []
#     tiempos_comparativos = {}

#     for molino in ["MC1", "MC2", "MC3"]:
#         # Verificar si el molino está apagado
#         if not verificar_estado_molino(molino):
#             print(f"El molino {molino} está apagado. No se generarán recomendaciones para este molino.")
#             continue  # Saltar molino apagado

#         # Obtener tiempos de vaciado para cada material
#         tiempos_vaciado = {}

#         for material in niveles_MC1_toneladas if molino == "MC1" else niveles_MC2_toneladas if molino == "MC2" else niveles_MC3_toneladas:
#             # Incluir solo los materiales correspondientes a cada molino
#             if (molino == "MC1" and material == "Puzolana_Humeda") or \
#                (molino == "MC2" and material in ["Puzolana_Humeda", "Puzolana_Seca"]) or \
#                (molino == "MC3" and material == "Puzolana_Seca"):
#                 tiempo_vaciado = calcular_tiempo_vaciado(molino, material)
#                 if tiempo_vaciado != float('inf'):
#                     tiempos_vaciado[material] = tiempo_vaciado
#             else:
#                 # Para otros materiales
#                 tiempo_vaciado = calcular_tiempo_vaciado(molino, material)
#                 if tiempo_vaciado != float('inf'):
#                     tiempos_vaciado[material] = tiempo_vaciado

#         # Guardar los tiempos para comparaciones entre molinos
#         for material, tiempo in tiempos_vaciado.items():
#             if material not in tiempos_comparativos:
#                 tiempos_comparativos[material] = {}
#             tiempos_comparativos[material][molino] = tiempo

#         # Generar recomendaciones para cada molino
#         recomendaciones, restricciones = generar_recomendaciones(molino)
#         recomendaciones_generales.extend(recomendaciones)
#         restricciones_generales.extend(restricciones)

#     # Comparar tiempos de vaciado entre molinos y seleccionar la mejor ruta de alimentación
#     print("\n--- Comparación de tiempos de vaciado entre molinos ---")

#     for material, tiempos in tiempos_comparativos.items():
#         print(f"\nMaterial: {material}")
#         for molino, tiempo in tiempos.items():
#             if tiempo is not None:  # Verifica si el tiempo no es None
#                 print(f"  {molino}: {tiempo:.2f} horas para vaciado")
#             else:
#                 print(f"  {molino}: Molino apagado")

#         # Seleccionar la ruta más crítica (el molino que necesita ser alimentado primero)
#         molino_mas_critico = min(tiempos, key=tiempos.get)
#         print(f"\nRuta más crítica para {material}: {molino_mas_critico}")

#         # Recomendar la alimentación a la ruta más crítica
#         ruta = seleccionar_ruta_alimentacion(molino_mas_critico, material)
#         recomendaciones_generales.append(f"Recomiendo alimentar {material} hacia {molino_mas_critico} usando la ruta: {ruta}")

#     return recomendaciones_generales, restricciones_generales

# def generar_recomendaciones_para_todos_los_molinos():
#     recomendaciones_generales = []
#     restricciones_generales = []
#     tiempos_comparativos = {}

#     for molino in ["MC1", "MC2", "MC3"]:
#         # Verificar si el molino está apagado
#         if not verificar_estado_molino(molino):
#             print(f"El molino {molino} está apagado. No se generarán recomendaciones para este molino.")
#             continue  # Saltar molino apagado

#         # Obtener tiempos de vaciado para cada material
#         tiempos_vaciado = {}

#         for material in niveles_MC1_toneladas if molino == "MC1" else niveles_MC2_toneladas if molino == "MC2" else niveles_MC3_toneladas:
#             # Incluir solo los materiales correspondientes a cada molino
#             if (molino == "MC1" and material in ["Puzolana_Humeda", "Clinker", "Yeso"]) or \
#                (molino == "MC2" and material in ["Puzolana_Humeda", "Puzolana_Seca", "Clinker", "Yeso"]) or \
#                (molino == "MC3" and material in ["Puzolana_Seca", "Clinker"]):
#                 tiempo_vaciado = calcular_tiempo_vaciado(molino, material)
#                 if tiempo_vaciado != float('inf'):
#                     tiempos_vaciado[material] = tiempo_vaciado

#         # Ordenar materiales por tiempos de vaciado
#         tiempos_vaciado_ordenados = sorted(tiempos_vaciado.items(), key=lambda item: item[1])

#         # Generar recomendaciones para cada material en el molino
#         for material, tiempo in tiempos_vaciado_ordenados:
#             # Verificar restricciones para cada material antes de recomendar su alimentación
#             validacion, mensaje = validar_restricciones(molino, material)
#             if validacion:
#                 recomendaciones_generales.append(f"Recomiendo alimentar {material} hacia {molino} usando la ruta: {seleccionar_ruta_alimentacion(molino, material)}")
#             else:
#                 restricciones_generales.append(mensaje)

#     return recomendaciones_generales, restricciones_generales

def revisar_tolvas_y_evitar_alimentacion(molino):
    rendimiento_MC1_MC2 = 1
    rendimiento_MC3 = 1

    if molino == "MC1":
        niveles = niveles_MC1
        tolvas = tolvas_MC1
        #return 0.2*tolvas[material]['max_metros']  # 20% de la capacidad
    elif molino == "MC2":
        niveles = niveles_MC2
        tolvas = tolvas_MC2
        #return 0.2*tolvas[material]['max_metros']  # 20% de la capacidad
    # else:  # MC3
    #     niveles = niveles_MC3
    #     tolvas = tolvas_MC3
    #     return 0.5*tolvas[material]['max_porcentaje'] # 50% de la capacidad
    elif molino == "MC3":
        # if material == "Clinker_Silo_Blanco":
        #     niveles = niveles_MC3
        #     tolvas = tolvas_MC3
        #  #   return 0.5*tolvas[material]['max_metros']  # 50% de la capacidad
        # else:
            niveles = niveles_MC3
            tolvas = tolvas_MC3
          #  return 0.5*tolvas[material]['max_porcentaje'] # 50% de la capacidad

    # Diccionario para almacenar el estado de cada tolva
    tolvas_llenas = {
        "MC1": {},
        "MC2": {},
        "MC3": {}
    }

    # # Verificación para MC1
    # for material, nivel in niveles_MC1.items():
    #     if nivel >= tolvas_MC1[material]['max_metros'] * rendimiento_MC1_MC2:
    #         print(f"niveles materiales",tolvas_MC1[material]['max_metros'])
    #         tolvas_llenas["MC1"][material] = True  # Tolva está llena, no permitir más alimentación
    #         print(f"Tolva de {material} en MC1 está llena con {nivel}t. No se permite más alimentación .")
    #     else:
    #         tolvas_llenas["MC1"][material] = False  # Tolva no está llena
    #         print(f" tolva de {material} no esta llena MC1,tiene un nivel de {nivel}m")

        # Verificación para MC1
    for material, niveles in niveles_MC1.items():
        if niveles >= tolvas_MC1[material]['max_metros'] * rendimiento_MC1_MC2:
            tolvas_llenas["MC1"][material] = True  # Tolva está llena, no permitir más alimentación
            print(f"Tolva de {material} en MC1 está llena con {niveles}t. No se permite más alimentación .")
        else:
            tolvas_llenas["MC1"][material] = False  # Tolva no está llena
            print(f" tolva de {material} no esta llena MC1,tiene un nivel de {niveles}m")

    # Verificación para MC2
    for material, niveles in niveles_MC2.items():
        if niveles >= tolvas_MC2[material]['max_metros'] * rendimiento_MC1_MC2:
            tolvas_llenas["MC2"][material] = True  # Tolva está llena, no permitir más alimentación
            print(f"Tolva de {material} en MC2 está llena. No se permite más alimentación.")
        else:
            tolvas_llenas["MC2"][material] = False  # Tolva no está llena

    # # Verificación para MC3, incluyendo Clinker silo blanco
    # for material, niveles in niveles_MC3.items():
    #     if 'max_metros' in tolvas[material]:
    #         if niveles >= tolvas_MC3[material]['max_metros'] * rendimiento_MC3:
    #             tolvas_llenas["MC3"][material] = True  # Tolva está llena, no permitir más alimentación
    #             print(f"Tolva de {material} en MC3 está llena. No se permite más alimentación.")
    #         else:
    #             tolvas_llenas["MC3"][material] = False  # Tolva no está llena
    #     else:
    #         if niveles >= tolvas_MC3[material]['max_porcentaje']:
    #           print(f"Tolva de {material} en MC3 está con porcentaje total. No se permite más alimentación.")

       # Verificación para MC3 (incluyendo materiales con 'max_metros' o 'max_porcentaje')
    for material, niveles in niveles_MC3.items():
        if 'max_metros' in tolvas_MC3[material]:
            if niveles >= tolvas_MC3[material]['max_metros'] * rendimiento_MC3:
                tolvas_llenas["MC3"][material] = True
                print(f"Tolva de {material} en MC3 está llena con {niveles}m. No se permite más alimentación.")
            else:
                tolvas_llenas["MC3"][material] = False
        elif 'max_porcentaje' in tolvas_MC3[material]:
            if niveles >= tolvas_MC3[material]['max_porcentaje'] * rendimiento_MC3:
                tolvas_llenas["MC3"][material] = True
                print(f"Tolva de {material} en MC3 está llena con {niveles}%. No se permite más alimentación.")
            else:
                tolvas_llenas["MC3"][material] = False

    return tolvas_llenas



def generar_recomendaciones_para_todos_los_molinos():
    recomendaciones_generales = []
    restricciones_generales = []
    tiempos_comparativos = {}

    for molino in ["MC1", "MC2", "MC3"]:
        # Verificar si el molino está apagado
        if not verificar_estado_molino(molino):
            print(f"El molino {molino} está apagado. No se generarán recomendaciones para este molino.")
            continue  # Saltar molino apagado

        # Obtener tiempos de vaciado para cada material
        tiempos_vaciado = {}

        # En MC1, queremos manejar la posibilidad de alimentar simultáneamente Clinker con Yeso o Puzolana
        if molino == "MC1":
            # Materiales críticos para MC1: Clinker, Yeso, Puzolana_Humeda
            materiales_criticos = ["Clinker", "Yeso", "Puzolana_Humeda"]
            for material in materiales_criticos:
                tiempo_vaciado = calcular_tiempo_vaciado(molino, material)
                if tiempo_vaciado != float('inf'):
                    tiempos_vaciado[material] = tiempo_vaciado
                    # Guardar tiempos para comparación entre molinos
                    if material not in tiempos_comparativos:
                        tiempos_comparativos[material] = {}
                    tiempos_comparativos[material][molino] = tiempo_vaciado

            # Ordenar materiales por tiempo de vaciado más crítico
            tiempos_vaciado_ordenados = sorted(tiempos_vaciado.items(), key=lambda item: item[1])

            #Seleccionar el material más crítico entre Yeso y Puzolana_Humeda
            material_critico = None
            for material, tiempo in tiempos_vaciado_ordenados:
                if material in ["Yeso", "Puzolana_Humeda"]:
                    material_critico = material
                    print(f"Material crítico encontrado ´´´´: {material}")
                    break
        else:
            # Ordenar materiales por tiempo de vaciado más crítico
            tiempos_vaciado_ordenados = sorted(tiempos_vaciado.items(), key=lambda item: item[1])
            #Seleccionar el material más crítico entre Yeso y Puzolana_Humeda
            material_critico = None
            for material, tiempo in tiempos_vaciado_ordenados:
                if material in ["Clinker","Yeso", "Puzolana_Humeda","Puzolana_Seca","Clinker_Silo_Blanco"]:
                    material_critico = material
                    print(f"Material crítico encontrado ´´´´: {material}")
                    break
            for material in niveles_MC2_toneladas if molino == "MC2" else niveles_MC3_toneladas:
                tiempo_vaciado = calcular_tiempo_vaciado(molino, material)
                if tiempo_vaciado != float('inf'):
                    tiempos_vaciado[material] = tiempo_vaciado
                    # Guardar tiempos para comparación entre molinos
                    if material not in tiempos_comparativos:
                        tiempos_comparativos[material] = {}
                    tiempos_comparativos[material][molino] = tiempo_vaciado

            # Generar recomendaciones normales para MC2 o MC3
            for material, tiempo in tiempos_vaciado.items():
                # if tiempo <= 2:
                    recomendaciones_generales.append(f"Recomiendo alimentar {material} en {molino}, quedan {tiempo:.2f} horas antes del vaciado.")
                # else:
                    # restricciones_generales.append(f"El tiempo de vaciado de {material} en {molino} es de {tiempo:.2f} horas. No es urgente alimentarlo ahora.")

    # Comparar tiempos de vaciado entre molinos y seleccionar la mejor ruta de alimentación
    print("\n--- Comparación de tiempos de vaciado entre molinos ---")

    for material, tiempos in tiempos_comparativos.items():
        print(f"\nMaterial: {material}")
        for molino, tiempo in tiempos.items():
            if tiempo is not None:  # Verifica si el tiempo no es None
                print(f"  {molino}: {tiempo:.2f} horas para vaciado")
            else:
                print(f"  {molino}: Molino apagado")

        # Seleccionar la ruta más crítica (el molino que necesita ser alimentado primero)
        molino_mas_critico = min(tiempos, key=tiempos.get)
        print(f"\nRuta más crítica para ---{material}: {molino_mas_critico}")

        # Recomendar la alimentación a la ruta más crítica
        ruta = seleccionar_ruta_alimentacion(molino_mas_critico, material,material_critico)
        recomendaciones_generales.append(f"Recomiendo alimentar {material} hacia {molino_mas_critico} usando la ruta: {ruta} ---")

    return recomendaciones_generales, restricciones_generales



# def generar_recomendaciones_para_todos_los_molinos():
#     recomendaciones_generales = []
#     restricciones_generales = []
#     tiempos_comparativos = {}

#     for molino in ["MC1", "MC2", "MC3"]:
#         # Verificar si el molino está apagado
#         if not verificar_estado_molino(molino):
#             print(f"El molino {molino} está apagado. No se generarán recomendaciones para este molino.")
#             continue  # Saltar molino apagado

#         # Obtener tiempos de vaciado para cada material
#         tiempos_vaciado = {}

#         # Definir materiales críticos según el molino
#         if molino == "MC1":
#             materiales_criticos = ["Clinker", "Yeso", "Puzolana_Humeda"]
#         elif molino == "MC2":
#             materiales_criticos = ["Clinker", "Yeso", "Puzolana_Humeda", "Puzolana_Seca"]
#         else:  # MC3
#             materiales_criticos = ["Clinker", "Yeso", "Puzolana_Seca"]

#         for material in materiales_criticos:
#             tiempo_vaciado = calcular_tiempo_vaciado(molino, material)
#             if tiempo_vaciado != float('inf'):
#                 tiempos_vaciado[material] = tiempo_vaciado
#                 # Guardar tiempos para comparación entre molinos
#                 if material not in tiempos_comparativos:
#                     tiempos_comparativos[material] = {}
#                 tiempos_comparativos[material][molino] = tiempo_vaciado

#         # Ordenar materiales por tiempo de vaciado más crítico
#         tiempos_vaciado_ordenados = sorted(tiempos_vaciado.items(), key=lambda item: item[1])

#         # Seleccionar el material más crítico para el molino actual
#         if tiempos_vaciado_ordenados:
#             material_critico = next(iter(tiempos_vaciado_ordenados))[0]
#             print(f"Material crítico en {molino}: {material_critico}")

#             # Recomendar la alimentación en función del material crítico
#             if molino == "MC1":
#                 # Verificar si es posible alimentar simultáneamente Clinker con el material crítico en MC1
#                 puede_alimentar_clinker = not ("Yeso" in alimentaciones_en_progreso.get(molino, []) or
#                                                "Puzolana_Humeda" in alimentaciones_en_progreso.get(molino, []))
#                 print(f"Puede alimentar simultáneamente Clinker con {material_critico}: {puede_alimentar_clinker} en {molino}")

#                 # Si se puede, recomendamos Clinker primero
#                 if puede_alimentar_clinker:
#                     ruta_clinker = seleccionar_ruta_clinker(molino)
#                     recomendaciones_generales.append(f"Recomiendo alimentar Clinker hacia {molino} usando la ruta: {ruta_clinker}")

#             # Generar la recomendación para el material crítico en el molino
#             ruta_material_critico = seleccionar_ruta_alimentacion(molino, material_critico,material_critico)
#             recomendaciones_generales.append(f"Recomiendo alimentar {material_critico} hacia {molino} usando la ruta: {ruta_material_critico}")

#         # Si no hay materiales críticos, no se recomiendan alimentaciones
#         else:
#             restricciones_generales.append(f"No se requiere alimentar ningún material crítico en {molino} en este momento.")

#     # Comparar tiempos de vaciado entre molinos y seleccionar la mejor ruta de alimentación
#     print("\n--- Comparación de tiempos de vaciado entre molinos ---")

#     for material, tiempos in tiempos_comparativos.items():
#         print(f"\nMaterial: {material}")
#         for molino, tiempo in tiempos.items():
#             if tiempo is not None:  # Verifica si el tiempo no es None
#                 print(f"  {molino}: {tiempo:.2f} horas para vaciado")
#             else:
#                 print(f"  {molino}: Molino apagado")

#         # Seleccionar la ruta más crítica (el molino que necesita ser alimentado primero)
#         molino_mas_critico = min(tiempos, key=tiempos.get)
#         print(f"\nRuta más crítica para ---{material}: {molino_mas_critico}")

#         # Recomendar la alimentación a la ruta más crítica usando el material crítico
#         ruta = seleccionar_ruta_alimentacion(molino_mas_critico, material, material_critico)
#         recomendaciones_generales.append(f"Recomiendo alimentar {material} hacia {molino_mas_critico} usando la ruta: {ruta}")

#     return recomendaciones_generales, restricciones_generales




def main():
    global niveles_MC1, niveles_MC2, niveles_MC3, tipos_produccion_actual
    ciclo = 1

    while True:
        print(f"\n--- Ciclo de optimización {ciclo} ---")

        actualizar_niveles_y_calcular_toneladas(niveles_MC1, niveles_MC2, niveles_MC3, tolvas_MC1,tolvas_MC2,tolvas_MC3)

        # Permitir al operador ingresar los niveles para cada molino
        print("\nIngresando niveles de materiales para MC1:")
        niveles_MC1 = ingresar_niveles("MC1")
        print("\nIngresando niveles de materiales para MC2:")
        niveles_MC2 = ingresar_niveles("MC2")
        print("\nIngresando niveles de materiales para MC3:")
        niveles_MC3 = ingresar_niveles("MC3")


        # Modificar el bloque principal para que solo optimice molinos en marcha
        for molino in ["MC1", "MC2", "MC3"]:
            # Verificar si el molino está apagado
            if not verificar_estado_molino(molino):
                #print(f"El molino {molino} está apagado. No se asignarán productos ni se realizarán cálculos.")
                continue  # Saltar el ciclo para molinos apagados

            # Definir los tipos de productos que cada molino puede producir
            if molino == "MC1":
                tipos_validos = ["P30", "P40"]
            elif molino == "MC2":
                tipos_validos = ["P10", "P16", "P20", "P30"]
            elif molino == "MC3":
                tipos_validos = ["P30"]

            print("")

            print(f"\nOptimizando {molino}")

            print(f"Este molino puede producir: {', '.join(tipos_validos)}")

            # Solicitar al usuario el tipo de producción para el molino
            tipo_produccion = input(f"Ingrese el tipo de producción para {molino} (o presione Enter para omitir): ").upper()

            if tipo_produccion:
                if tipo_produccion not in tipos_validos:
                    print(f"Error, Ingrese un tipo de producción válido para {molino}")
                    continue  # Si el tipo de producción no es válido, volver a solicitarlo

                tipos_produccion_actual[molino] = tipo_produccion  # Actualizar el tipo de producción en el diccionario
                print(f"Producto ingresado para {molino}: {tipo_produccion}")

                # Llamar a la función de optimización para el molino en marcha (sin tipo_produccion)
                optimizar_alimentacion(molino)

        # Generar recomendaciones de forma continua para todos los molinos
        recomendaciones_generales, restricciones_generales = generar_recomendaciones_para_todos_los_molinos()

        # Imprimir recomendaciones y restricciones
        print("\nRecomendaciones Generales:")
        for recomendacion in recomendaciones_generales:
            print(recomendacion)
        print("\nRestricciones Generales:")
        for restriccion in restricciones_generales:
            print(restriccion)


        ciclo += 1

        continuar = input("¿Desea continuar con otro ciclo? (s/n): ").lower()
        if continuar != 's':
            break

    print("Simulación completada.")

if __name__ == "__main__":
    main()



--- Ciclo de optimización 1 ---
--- Actualización de niveles y cálculo de toneladas ---
Niveles actuales en metros
Niveles MC1 = {'Clinker': 0, 'Puzolana_Humeda': 0, 'Yeso': 9}
Niveles MC2 = {'Clinker': 5, 'Puzolana_Humeda': 11, 'Puzolana_Seca': 7, 'Yeso': 6}
Niveles MC3 = {'Clinker': 78, 'Clinker_Silo_Blanco': 11, 'Puzolana_Seca': 64, 'Yeso': 0}

Niveles actuales en toneladas
Niveles MC1 toneladas = {'Clinker': 0.0, 'Puzolana_Humeda': 0.0, 'Yeso': 270.0}
Niveles MC2 toneladas = {'Clinker': 166.67, 'Puzolana_Humeda': 366.67, 'Puzolana_Seca': 58.33, 'Yeso': 80.0}
Niveles MC3 toneladas = {'Clinker': 46.8, 'Clinker_Silo_Blanco': 523.81, 'Puzolana_Seca': 22.4, 'Yeso': 0.0}


Ingresando niveles de materiales para MC1:
Ingrese el nivel actual de Clinker en MC1 (0 hasta 14 metros) (actual: 0 metros) o presione Enter para mantener el valor actual: 
Ingrese el nivel actual de Puzolana_Humeda en MC1 (0 hasta 12 metros) (actual: 0 metros) o presione Enter para mantener el valor actual: 
Ingrese 

KeyboardInterrupt: Interrupted by user